# Omni Anime Dubber v6

T4 x2 otimizado. Configure tudo na Celula 1.


## Cel 1 - Setup Global


In [ ]:
import time, os
import requests as http_requests

NOTEBOOK_NAME = "omni-anime-ver-final"
STEP_NAME = "step_omni"

# Carregar secrets
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    def _ks(n):
        try: return _s.get_secret(n)
        except: return ""
    DATABASE_URL = _ks("DATABASE_URL")
    PROJECT_ID = _ks("PIPELINE_PROJECT_ID")
    PIPELINE_WEBHOOK_URL = _ks("PIPELINE_WEBHOOK_URL")
except:
    DATABASE_URL = os.getenv("DATABASE_URL", "")
    PROJECT_ID = os.getenv("PIPELINE_PROJECT_ID", "")
    PIPELINE_WEBHOOK_URL = os.getenv("PIPELINE_WEBHOOK_URL", "")

_cell_timers = {}

def _db_exec(query, params):
    if not DATABASE_URL: return
    try:
        import psycopg2
        conn = psycopg2.connect(DATABASE_URL)
        cur = conn.cursor()
        cur.execute(query, params)
        conn.commit()
        cur.close()
        conn.close()
    except: pass

def cell_start(idx, name=""):
    _cell_timers[idx] = time.time()
    print(f"\n{'='*50}\n  CELULA [{idx}] {name}\n{'='*50}")
    if not PROJECT_ID: return
    _db_exec("INSERT INTO pipeline_cell_tracking (project_id,notebook,cell_index,cell_name,status,started_at) VALUES (%s::uuid,%s,%s,%s,'running',NOW()) ON CONFLICT DO NOTHING", (PROJECT_ID, NOTEBOOK_NAME, idx, name))
    _db_exec("UPDATE pipeline_cell_tracking SET status='running',started_at=NOW(),finished_at=NULL,cell_name=%s WHERE project_id=%s::uuid AND notebook=%s AND cell_index=%s", (name, PROJECT_ID, NOTEBOOK_NAME, idx))

def cell_end(idx, status="done", msg=""):
    elapsed = ""
    if idx in _cell_timers:
        s = int(time.time() - _cell_timers.pop(idx))
        elapsed = f" ({s//60}m{s%60}s)" if s >= 60 else f" ({s}s)"
    icon = "OK" if status == "done" else "ERRO"
    print(f"  [{icon}] CELULA [{idx}] {status}{elapsed}: {msg}\n{'_'*50}")
    if not PROJECT_ID: return
    _db_exec("UPDATE pipeline_cell_tracking SET status=%s,finished_at=NOW(),duration_seconds=EXTRACT(EPOCH FROM(NOW()-started_at)),message=%s WHERE project_id=%s::uuid AND notebook=%s AND cell_index=%s", (status, msg, PROJECT_ID, NOTEBOOK_NAME, idx))

def report_step(status, msg=""):
    if PROJECT_ID and PIPELINE_WEBHOOK_URL:
        try:
            http_requests.post(f"{PIPELINE_WEBHOOK_URL}/webhook/status", json={"project_id": PROJECT_ID, "step": STEP_NAME, "status": status, "message": msg}, timeout=15)
        except: pass
    print(f"\nNOTEBOOK FINALIZADO: {STEP_NAME} -> {status}")
    if not PROJECT_ID: return
    _db_exec(f"UPDATE pipeline_projects SET {STEP_NAME}=%s,current_step=%s,updated_at=NOW() WHERE id=%s::uuid", (status, STEP_NAME.replace("step_",""), PROJECT_ID))
    _db_exec("INSERT INTO pipeline_logs (project_id,step,status,message) VALUES (%s::uuid,%s,%s,%s)", (PROJECT_ID, STEP_NAME, status, msg))

print("Tracking de celulas configurado!")


In [ ]:
cell_start(1, "Celula 1")
# @title 🚀 1. Setup Inicial: Kaggle + Google Drive OAuth + Whisper + Modelos v6
# @markdown Configure TUDO aqui antes de rodar qualquer outra célula.

import os, sys, shutil, time, asyncio, json, nest_asyncio, subprocess, urllib.request, io, gc
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

# --- 1. INSTALAÇÃO DE DEPENDÊNCIAS ---
print("📦 Instalando bibliotecas (pode demorar 2-3 min)...")
os.system("apt-get install -y ffmpeg > /dev/null 2>&1")
os.system("pip install python-dotenv faster-whisper gradio_client google-genai edge-tts ffmpeg-python pydub openai nest_asyncio omnivoice torchaudio google-auth google-auth-httplib2 google-api-python-client stable-ts funasr modelscope demucs silero-vad onnxruntime pyannote.audio speechbrain scipy > /dev/null 2>&1")

import torch
import stable_whisper
from google import genai as google_genai_client
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from openai import OpenAI
from pydub import AudioSegment
from dotenv import load_dotenv
nest_asyncio.apply()

# ─────────────────────────────────────────────────────────────────────────────
# ⚙️  PAINEL CENTRAL DE CONFIGURAÇÃO — Edite aqui antes de rodar
# ─────────────────────────────────────────────────────────────────────────────

# 🎬 PRIORIDADE DE GERAÇÃO DE ÁUDIO
# Ambos (Video Completo + Short) serão sempre gerados.
# Este parâmetro define qual roda PRIMEIRO na Fábrica de Áudio (Célula 6).
TARGET_MODE_FIRST = "Video Completo"  # @param ["Video Completo", "Short Viral"]

# Gerar seleção viral + Guia SEO? (Célula 5)
GENERATE_SHORT = True  # @param {type:"boolean"}

# 🎯 DURAÇÃO ALVO DO SHORT (segundos)
SHORT_DURATION_TARGET = 60  # @param {type:"integer"}

# 🔊 AJUSTE DE VELOCIDADE DE ÁUDIO (Fábrica Cel 6)
TARGET_MIN_SPEED = 1.1
TARGET_MAX_SPEED = 1.35
MAX_AUDIO_RETRIES = 4
MAX_TEXT_RETRIES  = 3

# 🎙️ NARRADOR PADRÃO (usado para busca do áudio de clonagem no Drive)
# O sistema busca arquivos na pasta CLONAGEM/ do Drive cujo nome se pareça com esse.
# Ex: NOME_NARRADOR="Goku" vai achar "goku_dragon_ball.mp3" automaticamente.
NOME_NARRADOR = "Alessandro"  # @param {type:"string"}

# 📦 TRADUÇÃO — TAMANHO DO BATCH (blocos por chamada ao modelo)
# Se o anime foi identificado: usa BATCH_SIZE. Senão: usa 60 (mais conservador).
BATCH_SIZE = 50  # @param {type:"integer"}

# ─────────────────────────────────────────────────────────────────────────────
# 🤖 CADEIAS DE MODELOS GEMINI (fallback automático por quota/erro)
#    Adicione modelos extras separados por vírgula — serão tentados em ordem.
# ─────────────────────────────────────────────────────────────────────────────

# Identificação de anime — requer modelo com boa compreensão multilíngue
MODELS_IDENTIFICACAO = [
    "gemini-3.1-pro-preview",
    "gemini-3.1-flash-lite-preview",
    "gemini-3-flash-preview",
    "gemini-2.5-pro",
]

# Tradução em batch — velocidade é importante, flash é suficiente
MODELS_TRADUCAO = [
    "gemini-3.1-pro-preview",
    "gemini-3.1-flash-lite-preview",
    "gemini-2.5-pro",
    "gemini-3-flash-preview",
]

# Geração de Short e SEO — criatividade e copy
MODELS_SEO = [
    "gemini-3.1-pro-preview",
    "gemini-3.1-flash-lite-preview",
    "gemini-2.5-pro",
    "gemini-3-flash-preview",
]

# Reescrita de texto da fábrica de áudio — contagem de caracteres
MODELS_REESCRITA = [
    "gemini-3.1-flash-lite-preview",
    "gemini-3-flash-preview",
]

# Fallback para OpenAI caso TODOS os Gemini falhem
OPENAI_FALLBACK_MODEL = "gpt-5.4-nano"
USE_OPENAI_FALLBACK   = True  # @param {type:"boolean"}

# ─────────────────────────────────────────────────────────────────────────────
# 🔑 CHAVES DE API — Kaggle Secrets (com fallback para .env local)
# ─────────────────────────────────────────────────────────────────────────────
def _load_credentials():
    # Tenta Kaggle Secrets primeiro (ambiente de produção)
    try:
        from kaggle_secrets import UserSecretsClient
        _s = UserSecretsClient()
        def _get(name):
            try: return _s.get_secret(name)
            except: return ""
        print("Carregando chaves do Kaggle Secrets...")
        return _get
    except ImportError:
        pass

    # Fallback: .env local (ambiente de desenvolvimento)
    load_dotenv()
    print("Carregando chaves do .env (desenvolvimento local)...")
    return lambda name: os.getenv(name, "")

_get_secret = _load_credentials()

GEMINI_API_KEY      = _get_secret("GEMINI_API_KEY")
OPENAI_API_KEY      = _get_secret("OPENAI_API_KEY")
DRIVE_ACCESS_TOKEN  = _get_secret("DRIVE_ACCESS_TOKEN")
DRIVE_REFRESH_TOKEN = _get_secret("DRIVE_REFRESH_TOKEN")
DRIVE_CLIENT_ID     = _get_secret("DRIVE_CLIENT_ID")
DRIVE_CLIENT_SECRET = _get_secret("DRIVE_CLIENT_SECRET")
HF_TOKEN            = _get_secret("HF_TOKEN")

_keys_ok = True
for _name, _val in [("GEMINI_API_KEY", GEMINI_API_KEY), ("DRIVE_REFRESH_TOKEN", DRIVE_REFRESH_TOKEN)]:
    if not _val:
        print(f"  ATENCAO: '{_name}' nao encontrada! Verifique Kaggle Secrets.")
        _keys_ok = False
if _keys_ok:
    print("Todas as chaves carregadas com sucesso.")

# ─────────────────────────────────────────────────────────────────────────────
# 📁 CAMINHOS DO PROJETO
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH            = "/kaggle/working/AUDIO_DUB"
TEMP_PATH            = "/kaggle/working/temp_audio"
ANIME_AUDIO_ORIGINAL = "/kaggle/working/input/anime_audio.mp3"
MIN_GAP_CENA_S       = 0.3

MODELOS_OFFLINE       = ""
MODEL_WHISPER_PATH    = "large-v3"
SENSEVOICE_MODEL_PATH = "iic/SenseVoiceSmall"
MODEL_OMNIVOICE_PATH  = "k2-fsa/OmniVoice"

os.chdir("/kaggle/working")
for p in [BASE_PATH, TEMP_PATH, "/kaggle/working/input"]:
    os.makedirs(p, exist_ok=True)

print("Limpando arquivos temporarios...")
for f in os.listdir("/kaggle/working"):
    if f.endswith((".mp3", ".wav", ".srt")):
        try: os.remove(f"/kaggle/working/{f}")
        except: pass

# ─────────────────────────────────────────────────────────────────────────────
# ☁️  GOOGLE DRIVE
# ─────────────────────────────────────────────────────────────────────────────
print("Autenticando Google Drive...")
try:
    creds = Credentials(
        token=DRIVE_ACCESS_TOKEN if DRIVE_ACCESS_TOKEN else None,
        refresh_token=DRIVE_REFRESH_TOKEN,
        token_uri="https://oauth2.googleapis.com/token",
        client_id=DRIVE_CLIENT_ID,
        client_secret=DRIVE_CLIENT_SECRET,
        scopes=["https://www.googleapis.com/auth/drive"]
    )
    if creds.expired and creds.refresh_token:
        creds.refresh(Request())
    drive_service = build("drive", "v3", credentials=creds)
    print("Google Drive autenticado!")
except Exception as e:
    drive_service = None
    print(f"Falha na autenticacao do Drive: {e}")

def _buscar_id(caminho_no_drive):
    partes = caminho_no_drive.strip("/").split("/")
    parent_id = "root"
    for parte in partes:
        query = f"name='{parte}' and '{parent_id}' in parents and trashed=false"
        results = drive_service.files().list(q=query, fields="files(id, mimeType)").execute()
        arquivos = results.get("files", [])
        if not arquivos: return None
        parent_id = arquivos[0]["id"]
    return parent_id

def _garantir_pasta(caminho_pasta):
    partes = caminho_pasta.strip("/").split("/")
    parent_id = "root"
    for pasta in partes:
        query = f"name='{pasta}' and '{parent_id}' in parents and trashed=false and mimeType='application/vnd.google-apps.folder'"
        results = drive_service.files().list(q=query, fields="files(id)").execute()
        existentes = results.get("files", [])
        if existentes:
            parent_id = existentes[0]["id"]
        else:
            nova = drive_service.files().create(
                body={"name": pasta, "mimeType": "application/vnd.google-apps.folder", "parents": [parent_id]},
                fields="id"
            ).execute()
            parent_id = nova["id"]
    return parent_id

def baixar_do_drive(caminho_no_drive, destino_local):
    if os.path.exists(destino_local): return True
    try:
        file_id = _buscar_id(caminho_no_drive)
        if not file_id: return False
        request = drive_service.files().get_media(fileId=file_id)
        with open(destino_local, "wb") as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done: _, done = downloader.next_chunk()
        return True
    except: return False

def salvar_no_drive(caminho_local, caminho_destino_drive):
    if drive_service is None or not os.path.exists(caminho_local): return
    try:
        partes = caminho_destino_drive.strip("/").split("/")
        nome_arquivo = partes[-1]
        pasta_drive  = "/".join(partes[:-1]) if len(partes) > 1 else ""
        parent_id = _garantir_pasta(pasta_drive) if pasta_drive else "root"
        query = f"name='{nome_arquivo}' and '{parent_id}' in parents and trashed=false"
        results = drive_service.files().list(q=query, fields="files(id)").execute()
        existentes = results.get("files", [])
        media = MediaFileUpload(caminho_local, resumable=True)
        if existentes:
            drive_service.files().update(fileId=existentes[0]["id"], media_body=media).execute()
        else:
            drive_service.files().create(
                body={"name": nome_arquivo, "parents": [parent_id]},
                media_body=media, fields="id"
            ).execute()
        print(f"  Salvo no Drive: {caminho_destino_drive}")
    except Exception as e:
        print(f"  Erro ao salvar {caminho_destino_drive}: {e}")

# Download de cache do Drive
if drive_service:
    print("\nBaixando arquivos do Drive...")
    baixar_do_drive("KAGGLE/AUDIO_DUB/INPUT/anime_audio.mp3",      "/kaggle/working/input/anime_audio.mp3")
    baixar_do_drive("KAGGLE/AUDIO_DUB/transcricao_raw.json",        f"{BASE_PATH}/transcricao_raw.json")
    baixar_do_drive("KAGGLE/AUDIO_DUB/traducao_simplificada.json",  f"{BASE_PATH}/traducao_simplificada.json")
    baixar_do_drive("KAGGLE/AUDIO_DUB/identificacao_anime.json",    f"{BASE_PATH}/identificacao_anime.json")
    baixar_do_drive("KAGGLE/AUDIO_DUB/roteiro_short.json",          f"{BASE_PATH}/roteiro_short.json")

# --- Busca e download automático do áudio de referência para clonagem ---
REF_AUDIO_PATH = ""
REF_TEXT       = ""

if drive_service and NOME_NARRADOR:
    from difflib import SequenceMatcher
    def _sim(a, b): return SequenceMatcher(None, a.lower(), b.lower()).ratio()

    _clonagem_drive = "KAGGLE/AUDIO_DUB/INPUT/CLONAGEM"
    _clonagem_local = f"{BASE_PATH}/INPUT/CLONAGEM"
    os.makedirs(_clonagem_local, exist_ok=True)

    print(f"Buscando audio de clonagem para '{NOME_NARRADOR}'...")
    _pid = _buscar_id(_clonagem_drive)
    if _pid:
        _res = drive_service.files().list(
            q=f"'{_pid}' in parents and trashed=false and mimeType contains 'audio/'",
            fields="files(id, name)"
        ).execute()
        _melhor, _score = None, 0.0
        for _arq in _res.get('files', []):
            _s = _sim(NOME_NARRADOR, os.path.splitext(_arq['name'])[0])
            if NOME_NARRADOR.lower() in _arq['name'].lower(): _s = max(_s, 0.85)
            if _s > _score: _score, _melhor = _s, _arq
        if _melhor and _score > 0.6:
            print(f"  Match: '{_melhor['name']}' ({_score*100:.1f}%)")
            _local = f"{_clonagem_local}/{_melhor['name']}"
            if not os.path.exists(_local):
                _req = drive_service.files().get_media(fileId=_melhor['id'])
                with open(_local, "wb") as _f:
                    _dl = MediaIoBaseDownload(_f, _req)
                    _done = False
                    while not _done: _, _done = _dl.next_chunk()
            REF_AUDIO_PATH = _local
            print(f"  Referencia de clonagem definida: {REF_AUDIO_PATH}")
        else:
            print(f"  Nenhum match >60% para '{NOME_NARRADOR}' na pasta CLONAGEM.")
    else:
        print("  Pasta CLONAGEM nao encontrada no Drive.")

# ─────────────────────────────────────────────────────────────────────────────
# 🤖 CLIENTES DE API
# ─────────────────────────────────────────────────────────────────────────────
client_gemini = google_genai_client.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY.startswith("AIza") else None
client_openai = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY.startswith("sk-") else None

# ─────────────────────────────────────────────────────────────────────────────
# 🔁 FUNÇÃO CENTRAL DE CHAMADA GEMINI COM FALLBACK POR QUOTA
# ─────────────────────────────────────────────────────────────────────────────
# Erros que indicam quota esgotada ou sobrecarga — deve tentar próximo modelo
QUOTA_ERROR_CODES = {429, 503, 500}
QUOTA_ERROR_MSGS  = [
    "quota", "rate limit", "resource exhausted", "overloaded",
    "too many requests", "service unavailable", "capacity",
    "not found", "not supported", "is not found",
]

def _is_quota_error(e: Exception) -> bool:
    msg = str(e).lower()
    return any(kw in msg for kw in QUOTA_ERROR_MSGS)

def gemini_generate(model_chain: list, contents, config: dict = None, task_name: str = ""):
    """
    Tenta gerar conteúdo pelo Gemini percorrendo a cadeia de modelos.
    Em caso de erro de quota/sobrecarga, avança para o próximo.
    Erros fatais (autenticação, conteúdo inválido) param imediatamente.
    Retorna a resposta do primeiro modelo bem-sucedido.
    """
    if client_gemini is None:
        raise RuntimeError("client_gemini nao inicializado — verifique GEMINI_API_KEY no .env")

    last_exc = None
    for model in model_chain:
        try:
            kwargs = {"model": model, "contents": contents}
            if config:
                kwargs["config"] = config
            response = client_gemini.models.generate_content(**kwargs)
            print(f"  [OK] {task_name or 'gemini_generate'} -> {model}")
            return response
        except Exception as e:
            if _is_quota_error(e):
                print(f"  [QUOTA] {model}: {str(e)[:80]} — tentando proximo...")
                last_exc = e
                time.sleep(2)
                continue
            else:
                # Erro diferente de quota — relança imediatamente
                raise

    # Todos falharam — tenta OpenAI se habilitado
    if USE_OPENAI_FALLBACK and client_openai:
        print(f"  [FALLBACK] Todos os modelos Gemini falharam. Usando OpenAI {OPENAI_FALLBACK_MODEL}...")
        raise RuntimeError(
            f"OPENAI_FALLBACK: Todos Gemini esgotados. "
            f"Ultimo erro: {last_exc}. "
            f"Use client_openai com modelo {OPENAI_FALLBACK_MODEL} no bloco catch da celula chamadora."
        )

    raise RuntimeError(f"Todos os modelos Gemini falharam. Ultimo erro: {last_exc}")

# ─────────────────────────────────────────────────────────────────────────────
# 🎤 WHISPER (stable-ts + faster-whisper)
# ─────────────────────────────────────────────────────────────────────────────
GPU_COUNT    = torch.cuda.device_count()
device       = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if torch.cuda.is_available() else "int8"

print(f"\nGPUs disponíveis: {GPU_COUNT}")
print(f"Carregando Whisper em {device}:0 ({COMPUTE_TYPE})...")
if False:
    print("ERRO: Caminho do Whisper nao encontrado!")
else:
    stable_model = stable_whisper.load_faster_whisper(
        MODEL_WHISPER_PATH,
        device=device,
        device_index=0,        # GPU0 explícito (ctranslate2 não aceita "cuda:0")
        compute_type=COMPUTE_TYPE,
    )
    print("Whisper carregado em GPU0!")

# Preenchido pela Cel8 após subir servidores OmniVoice
OMNIVOICE_PORTS = []

# ─────────────────────────────────────────────────────────────────────────────
# 📋 RESUMO DA CONFIGURACAO
# ─────────────────────────────────────────────────────────────────────────────
_second = "Short Viral" if TARGET_MODE_FIRST == "Video Completo" else "Video Completo"
print(f"""
=================================================================
  CONFIGURACAO DO PIPELINE
=================================================================
  Ordem Fabrica   : 1) {TARGET_MODE_FIRST}  2) {_second}
  Gerar Short SEO : {GENERATE_SHORT} ({SHORT_DURATION_TARGET}s alvo)
  Velocidade alvo : {TARGET_MIN_SPEED}x - {TARGET_MAX_SPEED}x
  Fallback OpenAI : {USE_OPENAI_FALLBACK} ({OPENAI_FALLBACK_MODEL})
-----------------------------------------------------------------
  Identificacao   : {' > '.join(MODELS_IDENTIFICACAO)}
  Traducao        : {' > '.join(MODELS_TRADUCAO)}
  SEO/Short       : {' > '.join(MODELS_SEO)}
  Reescrita       : {' > '.join(MODELS_REESCRITA)}
=================================================================
""")

print("SETUP CONCLUIDO! Execute a Fase 5 (Servidores OmniVoice) antes de rodar a Fabrica de Audio.")

cell_end(1, "done", "Celula 1 concluido")

## Fase 1 - Transcricao

Demucs (GPU0) + Pyannote (GPU1) em paralelo.


In [ ]:
cell_start(2, "Setup Inicial + Drive + Whisper")
# =============================================================================
# @title 🎙️ 2. Transcrição v6 (Demucs + stable-ts + SenseVoice + Pyannote)
# @markdown Isolamento vocal, transcrição por região, classificação de idioma,
# @markdown segunda passagem para falas perdidas, e anti-alucinação.
# =============================================================================

import json, os, re, subprocess
import numpy as np
from pathlib import Path
from pydub import AudioSegment
from tqdm.notebook import tqdm
from funasr import AutoModel

# ─────────────────────────────────────────────────────────────────────────────
# ⚙️ CONFIGURAÇÕES
# ─────────────────────────────────────────────────────────────────────────────
TRANSCRICAO_PATH = f"{BASE_PATH}/transcricao_raw.json"

WHISPER_BEAM     = 5
MAX_SEG_WORDS    = 25

VAD_MIN_SILENCE_MS = 100
VAD_SPEECH_PAD_MS  = 60
VAD_THRESHOLD      = 0.25

VAD2_MIN_SILENCE_MS = 50
VAD2_SPEECH_PAD_MS  = 40
VAD2_THRESHOLD      = 0.15

VAD_TRIM_ONSET_PAD_MS  = 20
VAD_TRIM_OFFSET_PAD_MS = 30

MERGE_ZH_GAP_S   = 1.00
MERGE_JA_GAP_S   = 0.25
MIN_BLOCK_DUR_S  = 0.12
MAX_CHARS_PER_SEC = 18.0

SCAN_GAP_MIN_S    = 0.4
SCAN_SILERO_SENS  = 0.15

WATERMARK_PATTERNS = [
    "cảm ơn", "theo dõi", "hẹn gặp", "thank you for watching",
    "subscribe", "like and", "don't forget", "ご視聴", "チャンネル登録", "高評価",
    "다음 영상에서 만나요.", "ご視聴ありがとうございました", "また会いましょう", "지금까지 흔더리",
]

if not os.path.exists(ANIME_AUDIO_ORIGINAL):
    raise FileNotFoundError(f"❌ Áudio não encontrado: {ANIME_AUDIO_ORIGINAL}")

# =============================================================================
# 🔤 FUNÇÕES AUXILIARES
# =============================================================================
def analyze_script(text: str) -> dict:
    counts = {"hiragana": 0, "katakana": 0, "cjk": 0, "latin": 0, "other": 0}
    for ch in text:
        cp = ord(ch)
        if   0x3040 <= cp <= 0x309F: counts["hiragana"] += 1
        elif 0x30A0 <= cp <= 0x30FF: counts["katakana"] += 1
        elif (0x4E00 <= cp <= 0x9FFF) or (0x3400 <= cp <= 0x4DBF):
            counts["cjk"] += 1
        elif ch.isascii() and ch.isalpha(): counts["latin"] += 1
        elif not ch.isspace(): counts["other"] += 1
    total = sum(counts.values()) or 1
    kana  = counts["hiragana"] + counts["katakana"]
    if kana > 0:
        return {"lang": "ja", "confidence": min(1.0, 0.78 + (kana/total)*0.22)}
    if counts["cjk"] > 0:
        return {"lang": "zh", "confidence": min(1.0, 0.65 + (counts["cjk"]/total)*0.35)}
    if counts["latin"] > 0:
        return {"lang": "other", "confidence": 0.55}
    return {"lang": "ambiguous", "confidence": 0.25}

def is_hallucination(text: str, start: float, end: float) -> bool:
    dur = end - start
    text_lower = text.strip().lower()
    if any(pat in text_lower for pat in WATERMARK_PATTERNS): return True
    if dur > 0 and len(text) / dur > MAX_CHARS_PER_SEC: return True
    if len(text) <= 2 and dur < 0.4: return True
    return False

# =============================================================================
# 🔬 SILERO VAD
# =============================================================================
print("🔬 Carregando Silero VAD...")
try:
    from silero_vad import load_silero_vad, get_speech_timestamps
    _silero_model = load_silero_vad()
    SILERO_OK = True
    print("   ✅ Silero VAD pronto.")
except Exception:
    SILERO_OK = False
    print("   ⚠️  Silero VAD não disponível.")

def vad_trim(audio_full: AudioSegment, start_s: float, end_s: float) -> tuple:
    if not SILERO_OK: return start_s, end_s
    ms_s = max(0, int(start_s * 1000))
    ms_e = min(len(audio_full), int(end_s * 1000))
    clip = audio_full[ms_s:ms_e]
    if len(clip) < 80: return start_s, end_s
    clip_16k = clip.set_frame_rate(16000).set_channels(1)
    raw = np.array(clip_16k.get_array_of_samples(), dtype=np.float32) / 32768.0
    try:
        ts = get_speech_timestamps(raw, _silero_model, sampling_rate=16000, min_speech_duration_ms=50, min_silence_duration_ms=40)
    except Exception:
        return start_s, end_s
    if not ts: return start_s, end_s
    onset_ms  = max(0,        ts[0]["start"] / 16.0 - VAD_TRIM_ONSET_PAD_MS)
    offset_ms = min(len(clip), ts[-1]["end"]  / 16.0 + VAD_TRIM_OFFSET_PAD_MS)
    new_start = round(start_s + onset_ms  / 1000.0, 4)
    new_end   = round(start_s + offset_ms / 1000.0, 4)
    if new_end - new_start < 0.05: return start_s, end_s
    return new_start, new_end

# =============================================================================
# VERIFICACAO DE CACHE
# =============================================================================
if os.path.exists(TRANSCRICAO_PATH):
    print(f"Transcricao encontrada. Carregando cache...")
    with open(TRANSCRICAO_PATH, 'r', encoding='utf-8') as f:
        final_blocks = json.load(f)
    print(f"{len(final_blocks)} blocos carregados.")

else:
    # =====================================================================
    # 1/4 DEMUCS + PYANNOTE
    # 2 GPUs: Pyannote roda em GPU1 em paralelo com Demucs em GPU0.
    # 1 GPU:  sequencial (Demucs primeiro, depois Pyannote).
    # =====================================================================
    _gpu_count    = globals().get("GPU_COUNT", torch.cuda.device_count())
    _use_parallel = (_gpu_count >= 2)

    DIAR_SCRIPT = "/kaggle/working/_run_diar.py"
    DIAR_OUTPUT = "/kaggle/working/_diar_result.json"
    DIAR_WAV    = "/kaggle/working/_temp_diar.wav"

    # Pyannote/torchaudio tem bugs com MP3/M4A ("requested chunk resulted in x samples instead of y")
    # Resolvemos pre-convertendo para WAV mono 16k (tambem acelera o Pyannote brutalmente)
    subprocess.run(["ffmpeg", "-y", "-i", ANIME_AUDIO_ORIGINAL, "-ar", "16000", "-ac", "1", DIAR_WAV], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Gera o script de diarizacao usando lista de strings (evita conflito de aspas)
    _lines = [
        "import sys, types, torch, torchaudio\n",
        "if not hasattr(torchaudio, 'set_audio_backend'):\n",
        "    torchaudio.set_audio_backend = lambda x: None\n",
        "if not hasattr(torchaudio, 'get_audio_backend'):\n",
        "    torchaudio.get_audio_backend = lambda: 'soundfile'\n",
        "if 'torchaudio.backend' not in sys.modules:\n",
        "    sys.modules['torchaudio.backend'] = types.ModuleType('torchaudio.backend')\n",
        "if 'torchaudio.backend.common' not in sys.modules:\n",
        "    mod = types.ModuleType('torchaudio.backend.common')\n",
        "    mod.AudioMetaData = getattr(torchaudio, 'AudioMetaData', type('AudioMetaData', (), {}))\n",
        "    sys.modules['torchaudio.backend.common'] = mod\n",
        "import json\n",
        "from pyannote.audio import Pipeline\n",
        f"pipeline = Pipeline.from_pretrained('pyannote/speaker-diarization-3.1', token='{HF_TOKEN}')\n",
        "pipeline = pipeline.to(torch.device('cuda'))\n",
        f"diarization = pipeline('{DIAR_WAV}')\n",
        "if hasattr(diarization, 'speaker_diarization'): diarization = diarization.speaker_diarization\n",
        "segments = []\n",
        "for turn, _, speaker in diarization.itertracks(yield_label=True):\n",
        "    segments.append({'start': round(turn.start,4), 'end': round(turn.end,4), 'speaker': speaker})\n",
        f"with open('{DIAR_OUTPUT}', 'w') as out: json.dump(segments, out)\n",
        "print(f'Done {len(segments)} segmentos.', flush=True)\n",
    ]
    with open(DIAR_SCRIPT, "w", encoding="utf-8") as _df:
        _df.writelines(_lines)

    proc_diar = None
    if _use_parallel:
        _gpu1_env = {**os.environ, "CUDA_VISIBLE_DEVICES": "1"}
        print("   Pyannote iniciado em GPU1 (paralelo com Demucs)...")
        proc_diar = subprocess.Popen(
            ["python", DIAR_SCRIPT],
            env=_gpu1_env, stdout=subprocess.PIPE, stderr=subprocess.PIPE
        )
    else:
        print("   1 GPU detectada - Pyannote rodara apos Demucs.")

    _demucs_device = "cuda:0" if _use_parallel else "cuda"
    print(f"\n1/4  Demucs em {_demucs_device}...")
    subprocess.run(
        ["python", "-m", "demucs", "--two-stems", "vocals",
         "-n", "htdemucs", "-d", _demucs_device,
         "--out", "/kaggle/working/demucs_out", ANIME_AUDIO_ORIGINAL],
        check=True
    )
    stem_name        = Path(ANIME_AUDIO_ORIGINAL).stem
    VOCAL_PATH       = f"/kaggle/working/demucs_out/htdemucs/{stem_name}/vocals.wav"
    audio_full       = AudioSegment.from_file(VOCAL_PATH)
    total_duration_s = len(audio_full) / 1000.0
    print(f"   Vocais isolados: {total_duration_s:.1f}s")

    # ==========================================================================
    # 2/4 MODELOS AUXILIARES
    # ==========================================================================
    print("\n2/4  Carregando SenseVoice...")
    sense_model = AutoModel(model=SENSEVOICE_MODEL_PATH, device="cuda:0", disable_update=True)
    print("   SenseVoice Small carregado.")

    # Coleta ou roda Pyannote
    print("   Rodando/aguardando Pyannote...")
    if proc_diar is not None:
        _dout, _derr = proc_diar.communicate()   # aguarda e captura de uma vez
        _diar_ok = (proc_diar.returncode == 0)
        if not _diar_ok:
            print(f"   Pyannote (GPU1) falhou:\n{_derr.decode()[-1200:]}")
    else:
        _r = subprocess.run(["python", DIAR_SCRIPT], capture_output=True, text=True)
        _diar_ok = (_r.returncode == 0)
        if not _diar_ok:
            print(f"   Pyannote falhou:\n{_r.stderr[-1200:]}")

    if _diar_ok and os.path.exists(DIAR_OUTPUT):
        with open(DIAR_OUTPUT) as f:
            diar_segments = json.load(f)
        print(f"   {len(diar_segments)} segmentos de diarizacao coletados.")
    else:
        print("   Continuando sem agrupamento de speakers.")
        diar_segments = []


    def merge_diar_regions(diar_segs, gap_tol=0.4):
        if not diar_segs: return []
        s = sorted(diar_segs, key=lambda d: d["start"])
        merged = [dict(s[0])]
        for d in s[1:]:
            last = merged[-1]
            if d["speaker"] == last["speaker"] and (d["start"] - last["end"]) <= gap_tol:
                last["end"] = max(last["end"], d["end"])
            else:
                merged.append(dict(d))
        return merged

    speaker_regions = merge_diar_regions(diar_segments)
    full_regions = []
    cursor = 0.0
    for r in sorted(speaker_regions, key=lambda x: x["start"]):
        if r["start"] - cursor > 0.3:
            full_regions.append({"start": cursor, "end": r["start"], "speaker": "unknown"})
        full_regions.append(r)
        cursor = r["end"]
    if total_duration_s - cursor > 0.3:
        full_regions.append({"start": cursor, "end": total_duration_s, "speaker": "unknown"})

    # ==========================================================================
    # 🧠 TRANSCRIÇÃO POR REGIÃO
    # ==========================================================================
    TEMP_WAV = "/kaggle/working/_temp_region.wav"

    def transcribe_region(start_s, end_s, language=None, vad_min_silence_ms=VAD_MIN_SILENCE_MS, vad_speech_pad_ms=VAD_SPEECH_PAD_MS, threshold=VAD_THRESHOLD):
        ms_s = max(0, int(start_s * 1000))
        ms_e = min(len(audio_full), int(end_s * 1000))
        clip = audio_full[ms_s:ms_e]
        if len(clip) < 200: return []
        clip.export(TEMP_WAV, format="wav")
        try:
            if 'stable_model' not in dir(): raise RuntimeError('Whisper nao carregou. Verifique MODEL_WHISPER_PATH.')
            result = stable_model.transcribe_stable(
                TEMP_WAV, language=language, beam_size=WHISPER_BEAM, word_timestamps=True,
                vad_filter=True, vad_parameters=dict(min_silence_duration_ms=vad_min_silence_ms, speech_pad_ms=vad_speech_pad_ms, threshold=threshold),
                temperature=0, condition_on_previous_text=False, no_speech_threshold=0.3,
                log_prob_threshold=-1.5, compression_ratio_threshold=2.4,
                regroup=True, suppress_silence=True, only_voice_freq=True,
            )
        except Exception as e:
            print(f"      ⚠️ Erro [{start_s:.1f}-{end_s:.1f}s]: {e}")
            return []
        segs = []
        for seg in result.segments:
            text = (seg.text or "").strip()
            if not text: continue
            words = []
            if hasattr(seg, "words") and seg.words:
                for w in seg.words:
                    words.append({"word": w.word, "start": round(start_s + w.start, 4), "end": round(start_s + w.end, 4)})
            seg_start = words[0]["start"] if words else round(start_s + seg.start, 4)
            seg_end   = words[-1]["end"]  if words else round(start_s + seg.end,   4)
            seg_start = max(start_s, seg_start)
            seg_end   = min(end_s,   seg_end)
            if seg_end <= seg_start: continue
            segs.append({"text": text, "start": seg_start, "end": seg_end, "words": words, "avg_logprob": getattr(seg, "avg_logprob", 0.0), "detected_lang": result.language if hasattr(result, "language") else None})
        return segs

    # ==========================================================================
    # 🎤 3/4 TRANSCRIÇÃO
    # ==========================================================================
    print(f"\n🎤 3/4  Transcrevendo por região...")
    all_raw_segs = []
    for region in tqdm(full_regions, desc="Transcrevendo"):
        if region["end"] - region["start"] < 0.3: continue
        segs = transcribe_region(region["start"], region["end"])
        for s in segs:
            s["speaker"] = region["speaker"]
            all_raw_segs.append(s)
    print(f"   → {len(all_raw_segs)} segmentos brutos.")

    PHRASE_PUNCT = set("。！？…」』）")
    def split_long_segment(seg_words, max_words):
        if len(seg_words) <= max_words: return [seg_words]
        groups, current = [], []
        for w in seg_words:
            current.append(w)
            if any(ch in PHRASE_PUNCT for ch in w["word"].strip()) or len(current) >= max_words:
                groups.append(current); current = []
        if current:
            if groups and len(current) <= 3: groups[-1].extend(current)
            else: groups.append(current)
        return groups or [seg_words]

    # ==========================================================================
    # 🔀 4/4 CONSOLIDAÇÃO
    # ==========================================================================
    print("\n🧩 4/4  Consolidando...")
    TEMP_CHUNK = "/kaggle/working/_temp_chunk.wav"
    resolved = []
    for seg in tqdm(all_raw_segs, desc="Processando"):
        text = seg["text"].strip()
        if not text: continue
        words = seg.get("words", [])
        sub_groups = split_long_segment(words, MAX_SEG_WORDS) if words else [None]
        for grp in sub_groups:
            if grp:
                sub_text = "".join(w["word"] for w in grp).strip()
                sub_start, sub_end = grp[0]["start"], grp[-1]["end"]
            else:
                sub_text, sub_start, sub_end = text, seg["start"], seg["end"]
            if not sub_text or is_hallucination(sub_text, sub_start, sub_end): continue
            sub_start, sub_end = vad_trim(audio_full, sub_start, sub_end)
            if sub_end - sub_start < MIN_BLOCK_DUR_S: continue
            r = analyze_script(sub_text)
            if r["lang"] == "ja":
                lang_final, conf = "ja", r["confidence"]
            elif r["lang"] == "zh":
                audio_full[max(0,int(sub_start*1000)):min(len(audio_full),int(sub_end*1000))].export(TEMP_CHUNK, format="wav")
                try:
                    sv_res = sense_model.generate(input=TEMP_CHUNK, language="auto", use_itn=False)
                    m = re.search(r'<\|(zh|ja|yue|ko|en|nospeech)\|>', sv_res[0]["text"])
                    sv_tag = m.group(1) if m else "unknown"
                except Exception:
                    sv_tag = "unknown"
                if sv_tag == "ja" and (r["confidence"] < 0.8 or len(sub_text) < 6):
                    lang_final, conf = "ja", 0.65
                elif sv_tag in ("zh", "yue"):
                    lang_final, conf = "zh", min(1.0, r["confidence"] + 0.12)
                else:
                    lang_final, conf = "zh", r["confidence"] * 0.9
            else:
                lang_final, conf = "other", r["confidence"]
            tipo = "NARRACAO" if lang_final == "zh" else "CENA"
            resolved.append({"tipo": tipo, "start": round(sub_start, 4), "end": round(sub_end, 4), "text": sub_text, "lang_final": lang_final, "confidence": round(conf, 3), "logprob": round(seg.get("avg_logprob", 0.0), 3), "speaker": seg.get("speaker", "?")})
        resolved.sort(key=lambda b: b["start"])

    def merge_adjacent(blocks):
        if not blocks:
            return blocks
        # Conjunto de pontuação final de frase
        END_PUNCT = set(". ! ? 。！？…")
        out = [blocks[0]]
        for curr in blocks[1:]:
            prev = out[-1]
            gap = curr["start"] - prev["end"]
            prev_ends_punct = prev["text"].strip()[-1] in END_PUNCT if prev["text"].strip() else False

            if prev["tipo"] == "NARRACAO" == curr["tipo"] and gap <= MERGE_ZH_GAP_S and (prev["end"] - prev["start"]) < 5.0 and not prev_ends_punct:
                prev["text"] = prev["text"] if prev["text"] == curr["text"] else f"{prev['text']}{curr['text']}"
                prev["end"] = curr["end"]
                prev["confidence"] = (prev["confidence"] + curr["confidence"]) / 2
                continue

            if prev["tipo"] == "CENA" == curr["tipo"] and gap <= MERGE_JA_GAP_S and prev.get("speaker") == curr.get("speaker") and (prev["end"] - prev["start"]) < 2.0 and not prev_ends_punct:
                prev["text"] = f"{prev['text']}{curr['text']}"
                prev["end"] = curr["end"]
                continue

            out.append(curr)
        return out

    resolved = merge_adjacent(resolved)
    print(f"   → {len(resolved)} blocos após fusão.")

    # ==========================================================================
    # 🔍 3.5 SEGUNDA PASSAGEM
    # ==========================================================================
    print("\n🔍 3.5  Segunda passagem nos gaps...")
    all_times = sorted(resolved, key=lambda x: x["start"])
    gaps = []
    cursor = 0.0
    for blk in all_times:
        if blk["start"] - cursor >= SCAN_GAP_MIN_S: gaps.append((cursor, blk["start"]))
        cursor = max(cursor, blk["end"])
    if total_duration_s - cursor >= SCAN_GAP_MIN_S: gaps.append((cursor, total_duration_s))
    print(f"   → {len(gaps)} gaps.")

    new_segs = []
    for (gs, ge) in gaps:
        clip = audio_full[max(0,int(gs*1000)):min(len(audio_full),int(ge*1000))]
        if len(clip) < 200: continue
        clip_16k = clip.set_frame_rate(16000).set_channels(1)
        raw = np.array(clip_16k.get_array_of_samples(), dtype=np.float32) / 32768.0
        try:
            speech_ts = get_speech_timestamps(raw, _silero_model, sampling_rate=16000, min_speech_duration_ms=80, min_silence_duration_ms=60, threshold=SCAN_SILERO_SENS)
        except Exception:
            continue
        if not speech_ts: continue
        for sp in speech_ts:
            onset_s = gs + sp["start"] / 16000.0
            offset_s = gs + sp["end"] / 16000.0
            if offset_s - onset_s < 0.2: continue
            segs = transcribe_region(onset_s, offset_s, language="ja", vad_min_silence_ms=VAD2_MIN_SILENCE_MS, vad_speech_pad_ms=VAD2_SPEECH_PAD_MS, threshold=VAD2_THRESHOLD)
            for s in segs:
                if is_hallucination(s["text"], s["start"], s["end"]): continue
                r = analyze_script(s["text"])
                if r["lang"] == "ja" or (r["lang"] == "zh" and r["confidence"] < 0.8):
                    trim_s, trim_e = vad_trim(audio_full, s["start"], s["end"])
                    if trim_e - trim_s < MIN_BLOCK_DUR_S: continue
                    new_segs.append({"tipo": "CENA", "start": round(trim_s, 4), "end": round(trim_e, 4), "text": s["text"], "lang_final": "ja", "confidence": round(max(0.6, r["confidence"]), 3), "logprob": round(s.get("avg_logprob", 0.0), 3), "speaker": "unknown"})

    print(f"   → {len(new_segs)} novos segmentos.")
    resolved.extend(new_segs)
    resolved.sort(key=lambda b: b["start"])
    resolved = merge_adjacent(resolved)

    # Limpeza de watermarks
    HALL_PHRASES = {"ご視聴ありがとうございました", "ご視聴ありがとうございました。", "ご視聴ありがとうございます", "また会いましょう", "다음 영상에서 만나요.", "지금까지 흔더리", "Thanks for watching!", "Thank you for watching!", "チャンネル登録お願いします", "高評価お願いします", "please like and subscribe", "subscribe to my channel", "don't forget to subscribe", "to show"}
    before = len(resolved)
    resolved = [b for b in resolved if not (b["end"]-b["start"] < 2.5 and b.get("confidence",0) < 0.95 and any(p in b.get("text","") for p in HALL_PHRASES))]
    print(f"   🗑️  {before - len(resolved)} watermarks removidos.")

    # ==========================================================================
    # 🎬 TIMELINE FINAL
    # ==========================================================================


    # ==========================================================================
    # 🎬 TIMELINE FINAL
    # ==========================================================================

    final_blocks = []
    block_id = 0
    cursor = 0.0
    for seg in resolved:
        gap = seg["start"] - cursor
        if gap >= MIN_GAP_CENA_S:
            final_blocks.append({"id": block_id, "tipo": "CENA", "start": round(cursor, 4), "end": round(seg["start"], 4), "texto_original": "", "translated_text": "", "status": "CENA_SEM_FALA"})
            block_id += 1
        final_blocks.append({"id": block_id, "tipo": seg["tipo"], "start": round(seg["start"], 4), "end": round(seg["end"], 4), "texto_original": seg["text"], "translated_text": "", "status": "PENDING", "detected_lang": seg["lang_final"], "confidence": seg["confidence"], "avg_logprob": seg["logprob"], "speaker": seg.get("speaker", "?")})
        block_id += 1
        cursor = seg["end"]
    if total_duration_s - cursor >= MIN_GAP_CENA_S:
        final_blocks.append({"id": block_id, "tipo": "CENA", "start": round(cursor, 4), "end": round(total_duration_s, 4), "texto_original": "", "translated_text": "", "status": "CENA_SEM_FALA"})

    # ✅ PÓS-PROCESSAMENTO: CENA_SEM_FALA < 1s → descarta e estende o bloco anterior
    cleaned = []
    for blk in final_blocks:
        if blk.get("status") == "CENA_SEM_FALA" and (blk["end"] - blk["start"]) < 1.0:
            if cleaned:
                cleaned[-1]["end"] = blk["end"]
        else:
            cleaned.append(blk)
    for i, b in enumerate(cleaned):
        b["id"] = i
    final_blocks = cleaned

    with open(TRANSCRICAO_PATH, "w", encoding="utf-8") as f:
        json.dump(final_blocks, f, ensure_ascii=False, indent=4)

    n_nar = sum(1 for b in final_blocks if b["tipo"] == "NARRACAO")
    n_cena = sum(1 for b in final_blocks if b["tipo"] == "CENA" and b.get("status") == "PENDING")
    n_sil = sum(1 for b in final_blocks if b.get("status") == "CENA_SEM_FALA")
    print(f"\n✅ TRANSCRIÇÃO v6 CONCLUÍDA: {n_nar} NARRACAO | {n_cena} CENA c/ fala | {n_sil} CENA sem fala | {len(final_blocks)} total")
    print(f"💾 Salvo em: {TRANSCRICAO_PATH}")

# Preview
print("\n🔎 Preview (Top 10):")
for b in final_blocks[:10]:
    lp = f" [lp={b.get('avg_logprob', '')}]" if b.get("avg_logprob") else ""
    txt = b.get("texto_original", "")[:55]
    tag = f"[{b['tipo']}]" + (" [SEM FALA]" if b["status"] == "CENA_SEM_FALA" else "")
    print(f"  {tag} [{b['start']}s-{b['end']}s]{lp} {txt}")

if 'salvar_no_drive' in globals():
    print('💾 Salvando progresso no Drive...')
    salvar_no_drive(TRANSCRICAO_PATH, 'KAGGLE/AUDIO_DUB/transcricao_raw.json')

cell_end(2, "done", "Setup Inicial + Drive + Whisper concluido")

In [ ]:
cell_start(3, "Celula 3")
# @title 🕵️ 3. Identificação do Anime e Personagens
# @markdown Analisa o texto transcrito (zh/ja) para identificar o anime,
# @markdown protagonista e personagens antes da tradução.
# @markdown Modelos e fallback configurados na Célula 1.

import json
import os

IDENTIFICACAO_PATH = f"{BASE_PATH}/identificacao_anime.json"

if os.path.exists(IDENTIFICACAO_PATH):
    print("📁 Identificação já existe. Carregando cache...")
    with open(IDENTIFICACAO_PATH, 'r', encoding='utf-8') as f:
        anime_info = json.load(f)
    selected_anime = anime_info.get("title", "Desconhecido")
    protagonist    = anime_info.get("protagonist", "Narrador")
    print("✅ Cache carregado.")
else:
    if 'final_blocks' not in globals() or not final_blocks:
        try:
            with open(TRANSCRICAO_PATH, "r", encoding='utf-8') as f:
                final_blocks = json.load(f)
            print("⚠️ Transcrição carregada do arquivo.")
        except:
            raise Exception("❌ Rode a Célula 2 primeiro.")

    textos_com_fala = [b['texto_original'] for b in final_blocks
                       if b.get("texto_original", "").strip()]
    full_text_sample = " ".join(textos_com_fala[:60])

    if not full_text_sample.strip():
        raise Exception("❌ Nenhum texto original encontrado nos blocos.")

    print("🕵️ Analisando texto para identificar o anime...")

    identify_prompt = f"""
Analyze the following anime/movie transcript deeply.
The text may be in Chinese (Mandarin narration) and/or Japanese (character dialogue).

TEXT SAMPLE:
"{full_text_sample[:5000]}"

TASK:
1. Identify the **Anime/Movie Title** (use the most common English/Romaji name).
2. Identify the **main protagonist** — the character performing most actions in THIS specific text.
   - CRITICAL: Do not just pick the main franchise character.
   - Example: If the anime is "Naruto" but the text focuses on "Sasuke seeking revenge", pick Sasuke.
3. List ALL character names that appear or are referenced.
4. Brief synopsis of what happens in this specific segment.

OUTPUT FORMAT (JSON ONLY):
{{
    "title": "Anime Name",
    "title_jp": "Japanese Title (if known)",
    "protagonist": "Character Name",
    "characters": ["char1", "char2", "char3"],
    "synopsis": "Brief description of what happens",
    "reason": "Why you identified this anime and protagonist"
}}
"""

    anime_info = None
    try:
        # Tenta Gemini com fallback automático por quota (cadeias definidas na Cel1)
        response = gemini_generate(
            model_chain=MODELS_IDENTIFICACAO,
            contents=identify_prompt,
            config={'response_mime_type': 'application/json', 'temperature': 0.1},
            task_name="Identificacao"
        )
        anime_info = json.loads(response.text)

    except RuntimeError as e:
        # Todos Gemini falharam — tenta OpenAI se disponível
        if USE_OPENAI_FALLBACK and client_openai and "OPENAI_FALLBACK" in str(e):
            print("  Tentando OpenAI como fallback final...")
            try:
                resp = client_openai.chat.completions.create(
                    model=OPENAI_FALLBACK_MODEL,
                    messages=[{"role": "user", "content": identify_prompt}],
                    response_format={"type": "json_object"},
                    temperature=0.1
                )
                anime_info = json.loads(resp.choices[0].message.content)
                print("  OpenAI fallback OK.")
            except Exception as oe:
                print(f"  OpenAI também falhou: {oe}")

    except Exception as e:
        print(f"❌ Erro fatal na identificação: {e}")

    if not anime_info:
        anime_info = {
            "title": "Desconhecido",
            "protagonist": "Narrador",
            "characters": [],
            "synopsis": "",
            "reason": "Falha em todos os modelos disponíveis."
        }

    selected_anime = anime_info.get("title", "Desconhecido")
    protagonist    = anime_info.get("protagonist", "Narrador")

    with open(IDENTIFICACAO_PATH, 'w', encoding='utf-8') as f:
        json.dump(anime_info, f, ensure_ascii=False, indent=4)
    print(f"💾 Salvo: {IDENTIFICACAO_PATH}")

print(f"""
╔════════════════════════════════════════════════════════╗
║            🎬 ANIME IDENTIFICADO                       ║
╠════════════════════════════════════════════════════════╣
║  Título       : {selected_anime}
║  Protagonista : {protagonist}
║  Personagens  : {', '.join(anime_info.get('characters', [])[:8])}
║  Sinopse      : {anime_info.get('synopsis', 'N/A')[:80]}...
╚════════════════════════════════════════════════════════╝
""")

if 'salvar_no_drive' in globals():
    salvar_no_drive(IDENTIFICACAO_PATH, 'KAGGLE/AUDIO_DUB/identificacao_anime.json')

cell_end(3, "done", "Celula 3 concluido")

## Fase 2 - Traducao


In [ ]:
cell_start(4, "Transcricao Whisper + Pyannote")
# @title 🌍 4. Tradução Inteligente (com contexto do anime identificado)
# @markdown Traduz TODOS os blocos para português usando o nome do anime
# @markdown e personagens como contexto para evitar erros de nomes/termos.
# @markdown
# @markdown  NARRACAO (zh): Forçado em 3ª pessoa, mesmo que o original seja 1ª.
# @markdown  CENA (ja): Preserva a pessoa gramatical do original (1ª ou 3ª).
# @markdown Modelos e fallback configurados na Célula 1.

import json
import time
import os

TARGET_LANGUAGE  = "Portuguese (Brazil)"

TRADUCAO_PATH    = f"{BASE_PATH}/traducao_simplificada.json"
TRANSCRICAO_PATH = f"{BASE_PATH}/transcricao_raw.json"

_anime_title = globals().get('selected_anime', 'Desconhecido')
_protagonist  = globals().get('protagonist', 'Narrador')
_characters   = anime_info.get('characters', []) if 'anime_info' in globals() else []
_char_list    = ', '.join(_characters) if _characters else 'Desconhecidos'

print(f"Contexto: {_anime_title} | Protagonista: {_protagonist}")
print(f"Personagens: {_char_list}")

if os.path.exists(TRADUCAO_PATH):
    print("Traducao ja existe. Carregando cache...")
    with open(TRADUCAO_PATH, 'r', encoding='utf-8') as f:
        final_blocks = json.load(f)
    print(f"{len(final_blocks)} blocos carregados.")
else:
    if 'final_blocks' not in globals() or not final_blocks:
        try:
            with open(TRANSCRICAO_PATH, "r", encoding='utf-8') as f:
                final_blocks = json.load(f)
            print("Backup carregado do arquivo.")
        except:
            raise Exception("Rode a Celula 2 primeiro.")

    blocos_para_traduzir = [
        b for b in final_blocks
        if b.get("texto_original", "").strip() and not b.get("translated_text", "").strip()
    ]
    blocos_cena_sem_fala = [
        b for b in final_blocks
        if b["tipo"] == "CENA" and not b.get("texto_original", "").strip()
    ]

    for b in blocos_cena_sem_fala:
        b["translated_text"] = ""
        b["status"] = "CENA_SEM_FALA"

    narracao_list  = [b for b in blocos_para_traduzir if b["tipo"] == "NARRACAO"]
    cena_fala_list = [b for b in blocos_para_traduzir if b["tipo"] == "CENA"]

    total_lines = len(blocos_para_traduzir)
    print(f"Traduzindo {total_lines} blocos | {len(narracao_list)} narracoes | {len(cena_fala_list)} falas de cena")

    # ──────────────────────────────────────────────────────────────────────────
    # PROMPTS SEPARADOS POR TIPO
    # ──────────────────────────────────────────────────────────────────────────

    SYSTEM_NARRACAO = f"""
You are a professional dubbing scriptwriter specializing in anime narration for viral short-form videos.
Your task is to translate Chinese Mandarin narration blocks into {TARGET_LANGUAGE}.

CONTEXT: Anime "{_anime_title}" | Main character: {_protagonist} | Known cast: {_char_list}.

─── TRANSLATION RULES ───────────────────────────────────────────────────────

1. PERSON — Always Third Person. Never First Person.
   - "我" / "我们" → "ele/ela/eles", never "eu/nós".
   - Example: "我看到了敌人" → "Ele avistou o inimigo" ✅ | "Eu vi o inimigo" ❌

2. LANGUAGE LEVEL — Simple, clear, everyday vocabulary (A2/B1).
   - Write for the EAR, not the page. Short sentences, natural spoken rhythm.
   - Avoid academic, archaic, or overly formal words.

3. REPETITION — Never repeat the same word or phrase twice in a row across consecutive lines.
   - Vary pronouns and synonyms naturally.

4. PROPER NOUNS — Only keep names that are actual anime character names from the cast list above.
   - Kinship/role words are NOT proper nouns: "pai", "mãe", "mestre", "rei", "imperador", etc.
     Translate them normally — do NOT keep "Father", "Master", "Lord" in English mid-sentence.
   - Example: "Father said..." → "Seu pai disse..." ✅ | "Father disse..." ❌

5. SYNC — Keep the translated length close to the original for audio timing.
   One block = one line. Do not merge or split blocks.

─── OUTPUT ──────────────────────────────────────────────────────────────────

Respond with JSON only. No explanation, no markdown, no extra keys.
{{"translations": ["string 1", "string 2", ...]}}
The array MUST contain exactly {{total}} items in the original order.
"""

    SYSTEM_CENA = f"""
You are a professional dubbing scriptwriter for viral anime videos.
Translate the input list from Japanese into {TARGET_LANGUAGE}.

CONTEXT: Anime "{_anime_title}" | Main character: {_protagonist} | Known cast: {_char_list}.

TRANSLATION RULES FOR DIALOGUE BLOCKS (Character Dialogue):
1. PERSON: Preserve the grammatical person of the original speaker.
   - If the character speaks in First Person (僕/俺/私 = I/me), keep it as First Person in PT-BR.
   - If the character narrates in Third Person, keep Third Person.
   - DO NOT forcefully change "Eu" to "Ele" — these are authentic character voices.
2. AUTHENTICITY: Keep the character's voice, tone and personality.
   - Aggressive character → speak with aggression.
   - Timid character → speak softly/hesitantly.
3. VOCABULARY: Colloquial Brazilian Portuguese. Natural speech, not literary.
4. SYNC: Keep length similar to the original for lip-sync timing.
5. NAMES: Never translate proper nouns. Use the correct anime character names.

OUTPUT FORMAT (JSON ONLY):
{{ "translations": ["string 1", "string 2", ...] }}
The list MUST have exactly {{total}} items in the same order.
"""

    # ──────────────────────────────────────────────────────────────────────────
    # FUNÇÃO DE TRADUÇÃO COM FALLBACK
    # ──────────────────────────────────────────────────────────────────────────
    def _traduzir_chunk(chunk, system_instruction, task_label):
        """Traduz um chunk de blocos com fallback. Retorna lista de traduções ou None."""
        source_texts = [b["texto_original"] for b in chunk]
        total = len(source_texts)
        user_payload = json.dumps({"sentences": source_texts}, ensure_ascii=False)
        full_prompt  = f"{system_instruction}\n\nINPUT DATA:\n{user_payload}"
        translated_list = None
        try:
            response = gemini_generate(
                model_chain=MODELS_TRADUCAO,
                contents=full_prompt,
                config={'response_mime_type': 'application/json', 'temperature': 0.2},
                task_name=task_label
            )
            data = json.loads(response.text)
            if isinstance(data, list): translated_list = data
            elif "translations" in data: translated_list = data["translations"]
            else: translated_list = list(data.values())[0]
        except RuntimeError as e:
            if USE_OPENAI_FALLBACK and client_openai and "OPENAI_FALLBACK" in str(e):
                try:
                    resp = client_openai.chat.completions.create(
                        model=OPENAI_FALLBACK_MODEL,
                        messages=[
                            {"role": "system", "content": system_instruction},
                            {"role": "user",   "content": user_payload}
                        ],
                        response_format={"type": "json_object"}, temperature=0.2
                    )
                    data = json.loads(resp.choices[0].message.content)
                    translated_list = data.get("translations", [])
                    print(f"  OpenAI fallback OK ({task_label}).")
                except Exception as oe:
                    print(f"  OpenAI tambem falhou: {oe}")
        except Exception as e:
            print(f"Erro ({task_label}): {e}")
        return translated_list, total

    def traduzir_batch(blocks, system_instruction_template, task_label):
        if not blocks: return
        # Determina tamanho do chunk: se anime identificado usa BATCH_SIZE, senão 60
        _anime_ok = globals().get('selected_anime', 'Desconhecido') != 'Desconhecido'
        _chunk_sz  = globals().get('BATCH_SIZE', 50) if _anime_ok else 60
        total = len(blocks)
        system_instruction = system_instruction_template.replace("{total}", str(_chunk_sz))
        print(f"  Chunks de {_chunk_sz} | Total: {total} blocos")

        for i in range(0, total, _chunk_sz):
            chunk = blocks[i: i + _chunk_sz]
            chunk_label = f"{task_label}-chunk{i//  _chunk_sz + 1}"
            # Ajusta {total} para o tamanho real do chunk
            sys_inst = system_instruction_template.replace("{total}", str(len(chunk)))
            translated_list, expected = _traduzir_chunk(chunk, sys_inst, chunk_label)

            if not translated_list:
                for block in chunk:
                    block["translated_text"] = f"[FALHA] {block['texto_original']}"
                    block["status"] = "FAILED"
                continue

            if len(translated_list) != expected:
                print(f"  Recebidos {len(translated_list)} vs esperados {expected}. Ajustando...")

            for j, block in enumerate(chunk):
                block["translated_text"] = (
                    translated_list[j] if j < len(translated_list)
                    else f"[FALHA] {block['texto_original']}"
                )
                block["status"] = "TRANSLATED"


            print(f"  Chunk {i//  _chunk_sz + 1}: {len(chunk)} blocos traduzidos.")

    # ──────────────────────────────────────────────────────────────────────────
    # EXECUÇÃO
    # ──────────────────────────────────────────────────────────────────────────
    start = time.time()

    if narracao_list:
        print(f"\nTraduzindo {len(narracao_list)} narracoes (zh -> pt, 3a pessoa forcada)...")
        traduzir_batch(narracao_list, SYSTEM_NARRACAO, "Narracao-zh")

    if cena_fala_list:
        print(f"\nTraduzindo {len(cena_fala_list)} falas de cena (ja -> pt, pessoa original)...")
        traduzir_batch(cena_fala_list, SYSTEM_CENA, "Cena-ja")

    elapsed = time.time() - start
    print(f"\nTraducao concluida em {elapsed:.2f}s.")

    with open(TRADUCAO_PATH, 'w', encoding='utf-8') as f:
        json.dump(final_blocks, f, ensure_ascii=False, indent=4)
    print(f"Salvo: {TRADUCAO_PATH}")

    print("\nPreview (3 narracoes):")
    print("-" * 50)
    for b in [x for x in final_blocks if x["tipo"] == "NARRACAO"][:3]:
        print(f"Original : {b['texto_original']}")
        print(f"Traduzido: {b['translated_text']}")
        print("-" * 50)

    print("\nPreview (3 falas de cena):")
    print("-" * 50)
    for b in [x for x in final_blocks if x["tipo"] == "CENA" and x.get("translated_text")][:3]:
        print(f"Original : {b['texto_original']}")
        print(f"Traduzido: {b['translated_text']}")
        print("-" * 50)

if 'salvar_no_drive' in globals():
    salvar_no_drive(TRADUCAO_PATH, 'KAGGLE/AUDIO_DUB/traducao_simplificada.json')

cell_end(4, "done", "Transcricao Whisper + Pyannote concluido")

## Fase 3 - Short + SEO


In [ ]:
cell_start(5, "Identificacao do Anime")
# @title 🚀 5. Gerador Viral: Short & Guia de Postagem
# @markdown Seleciona o segmento mais viral e gera metadados SEO completos.
# @markdown TARGET_MODE e GENERATE_SHORT definidos na Celula 1.

import json
import copy
from datetime import timedelta
import os

if 'short_blocks' in globals():
    del short_blocks
short_blocks = []

# Lê configuração global da Cel1
_generate_short   = globals().get('GENERATE_SHORT', True)
_short_target_dur = globals().get('SHORT_DURATION_TARGET', 60)
current_anime_title = globals().get('selected_anime', 'Anime Desconhecido')
current_protagonist = globals().get('protagonist', 'Narrador')

SHORT_PATH = f"{BASE_PATH}/roteiro_short.json"
GUIA_PATH  = f"{BASE_PATH}/GUIA_VIRAL_{current_anime_title.replace(' ', '_')}.txt"

def fmt_timestamp(seconds):
    td = timedelta(seconds=int(seconds))
    return str(td)[2:]

# ──────────────────────────────────────────────────────────────────────────────
# CARREGAMENTO DOS BLOCOS
# ──────────────────────────────────────────────────────────────────────────────
if 'final_blocks' not in globals() or not final_blocks:
    for candidate in [
        f"{BASE_PATH}/traducao_simplificada.json",
        f"{BASE_PATH}/transcricao_raw.json",
    ]:
        if os.path.exists(candidate):
            with open(candidate, 'r', encoding='utf-8') as f:
                final_blocks = json.load(f)
            print(f"Roteiro carregado: {candidate}")
            break
    else:
        final_blocks = []

full_story_context = " ".join([
    b['translated_text'] for b in final_blocks
    if b.get('translated_text', '').strip()
])

# ──────────────────────────────────────────────────────────────────────────────
# PARTE A: GERADOR DE SHORT
# ──────────────────────────────────────────────────────────────────────────────
short_data_report = ""

if _generate_short:
    print(f"Analisando roteiro para encontrar o minuto de ouro (~{_short_target_dur}s)...")

    if not final_blocks:
        raise Exception("Sem blocos para processar. Rode a Celula 4 primeiro.")

    script_analysis = []
    for b in final_blocks:
        if not b.get('translated_text', '').strip():
            continue
        char_count = len(b['translated_text'])
        line = (
            f"ID:{b['id']} | T:{b['start']:.1f}-{b['end']:.1f} | "
            f"TIPO:{b['tipo']} | LIMIT:{int(char_count*0.85)}-{int(char_count*1.15)}c | "
            f"TXT:{b['translated_text']}"
        )
        script_analysis.append(line)

    short_prompt = f"""
ROLE: Elite Viral Video Editor specialized in anime content for TikTok/Shorts/Reels.

ANIME: "{current_anime_title}"
PROTAGONIST: {current_protagonist}

FULL SCRIPT CONTENT (context):
"{full_story_context[:6000]}"

TASK: Extract the most VIRAL continuous segment (~{_short_target_dur}s) from the script data below.

SELECTION CRITERIA (priority order):
1. TENSION PEAK: Highest emotional intensity — betrayal, revelation, power-up, death, sacrifice.
2. HOOK POTENTIAL: Starting block must have an instantly gripping opening line.
3. CLIFFHANGER: Ending block must cut at maximum curiosity — just before resolution.
4. DIALOGUE DENSITY: Prefer NARRACAO + CENA mix over pure narration.
5. VISUAL APPEAL: Fight scenes, emotional moments, dramatic reveals.

INSTRUCTIONS:
- Segment MUST be continuous (consecutive IDs).
- Hook: Rewrite first block to start with the conflict, not backstory.
- Cliffhanger: Rewrite last block to end at peak curiosity.
- NARRATION blocks: Always Third Person (Ele/Ela).
- DIALOGUE blocks: Preserve the character's original person (1st or 3rd).
- Respect character limits shown.

SCRIPT DATA:
{chr(10).join(script_analysis)}

OUTPUT JSON ONLY:
{{
    "start_id": <int>,
    "end_id": <int>,
    "hook_text": "<rewritten opening>",
    "cliffhanger_text": "<rewritten ending>",
    "tension_score": <1-10>,
    "reasoning": "<why this segment>"
}}
"""

    data = None
    try:
        response = gemini_generate(
            model_chain=MODELS_SEO,
            contents=short_prompt,
            config={'response_mime_type': 'application/json', 'temperature': 0.8},
            task_name="Short-Selection"
        )
        data = json.loads(response.text)

    except RuntimeError as e:
        if USE_OPENAI_FALLBACK and client_openai and "OPENAI_FALLBACK" in str(e):
            print("  Tentando OpenAI fallback para Short...")
            try:
                resp = client_openai.chat.completions.create(
                    model=OPENAI_FALLBACK_MODEL,
                    messages=[{"role": "user", "content": short_prompt}],
                    response_format={"type": "json_object"},
                    temperature=0.8
                )
                data = json.loads(resp.choices[0].message.content)
                print("  OpenAI fallback OK.")
            except Exception as oe:
                print(f"  OpenAI tambem falhou: {oe}")

    except Exception as e:
        print(f"Erro no Short: {e}")

    if data:
        s_id = data["start_id"]
        e_id = data["end_id"]
        idx_start = next((i for i, b in enumerate(final_blocks) if b["id"] == s_id), 0)
        idx_end   = next((i for i, b in enumerate(final_blocks) if b["id"] == e_id), 0)

        short_blocks = copy.deepcopy(final_blocks[idx_start: idx_end + 1])
        short_blocks[0]['translated_text']  = data['hook_text']
        short_blocks[-1]['translated_text'] = data['cliffhanger_text']

        t_start = fmt_timestamp(short_blocks[0]['start'])
        t_end   = fmt_timestamp(short_blocks[-1]['end'])

        short_data_report = f"""
# ROTEIRO DO SHORT
Minutagem  : {t_start} ate {t_end}
Tensao     : {data.get('tension_score', '?')}/10
Logica     : {data.get('reasoning', '')}
Hook       : {data['hook_text']}
Cliffhanger: {data['cliffhanger_text']}
"""
        print(f"Short Gerado! ({t_start} - {t_end}) | Tensao: {data.get('tension_score', '?')}/10")

        with open(SHORT_PATH, "w", encoding='utf-8') as f:
            json.dump(short_blocks, f, ensure_ascii=False, indent=4)

        if 'salvar_no_drive' in globals():
            salvar_no_drive(SHORT_PATH, "KAGGLE/AUDIO_DUB/roteiro_short.json")
    else:
        print("Short nao gerado (falha em todos os modelos).")
        short_data_report = "ERRO NA GERACAO."
else:
    print("Short desativado (GENERATE_SHORT=False na Celula 1).")
    short_data_report = ""

# ──────────────────────────────────────────────────────────────────────────────
# PARTE B: GUIA DE POSTAGEM SEO
# ──────────────────────────────────────────────────────────────────────────────
print("\nGerando Guia de Postagem SEO...")

seo_prompt = f"""
ACT AS: Senior YouTube & TikTok Growth Strategist with 10M+ subscribers managed.
ANIME: {current_anime_title}
PROTAGONIST: {current_protagonist}

FULL SCRIPT (analyze for keywords, emotions, plot points):
"{full_story_context[:5000]}"

TASK: Create HIGH-CTR metadata optimized for the YouTube and TikTok algorithms.
LANGUAGE: Portuguese (Brazil) — use colloquial, emotional, urgent tone.
NO GENERIC INTROS. Start every title with the conflict or mystery.

OUTPUT FORMAT (text only, follow EXACTLY):

# DADOS DO VIDEO
**Anime:** {current_anime_title}
**Sinopse:** [1-2 sentence emotional summary — focus on the CONFLICT, not the setup]

---

# YOUTUBE (Video Completo)

**TITULOS (5 opcoes — Clickbait Etico):**
1. [MISTERIO: Pergunta que gera curiosidade sobre o plot twist]
2. [PODER: Foco na transformacao/poder do protagonista]
3. [TRAICAO: Foco no conflito emocional entre personagens]
4. [NUMERO: "Os 3 momentos que mudaram..." ou similar]
5. [POLEMICA: Opiniao controversa mas verdadeira sobre o anime]

**DESCRICAO (Modelo AIDA — 4 paragrafos):**
Paragrafo 1 (ATENCAO): Comece com uma pergunta retorica bombástica sobre o conflito principal.
Paragrafo 2 (INTERESSE): Revele um detalhe surpreendente do enredo sem dar spoiler total.
Paragrafo 3 (DESEJO): Explique o que o espectador vai sentir assistindo.
Paragrafo 4 (ACAO): CTA direto — "Se inscreva pra nao perder a parte 2" + mencao ao like.

**TAGS (30 tags — mix estrategico):**
[5 tags broad: anime, dublado, etc]
[10 tags especificas do anime: nome, personagens, arcos]
[10 tags long-tail: "anime onde o protagonista...", frases de busca]
[5 tags trending: termos populares do momento]

---

# TIKTOK / SHORTS / REELS

**HEADLINES (5 opcoes):**
1. [PERGUNTA provocativa]
2. [AFIRMACAO chocante]
3. [POV: voce descobre que...]
4. [DESAFIO: Ninguem aguenta assistir sem chorar]
5. [GATILHO: A cena que traumatizou todo mundo]

**LEGENDA:** [Texto curto emocional + 3-5 hashtags virais no corpo]

**HASHTAGS (15 — ordenadas por potencial viral):**
[5 hashtags gerais de anime com alto volume]
[5 hashtags especificas do anime/personagem]
[5 hashtags de tendencia/formato: animedublado, animetiktok, etc]

---

# SUGESTAO DE THUMBNAIL
**Imagem:** [Descreva a cena ideal — expressao facial, acao, fundo]
**Texto overlay:** [Maximo 4 palavras em caps]
**Cores:** [Esquema de cores que contrasta com o feed]
**Expressao:** [Choque, raiva, tristeza — a emocao que gera clique]

---

# MELHOR HORARIO DE POSTAGEM
**YouTube:** [Dia e horario ideal para anime em pt-BR]
**TikTok:** [Dia e horario ideal]
**Motivo:** [Justificativa baseada no publico-alvo]
"""

seo_content = None
seo_model_used = ""

try:
    response = gemini_generate(
        model_chain=MODELS_SEO,
        contents=seo_prompt,
        config={'temperature': 0.9},
        task_name="SEO-Guia"
    )
    seo_content = response.text
    seo_model_used = "Gemini"

except RuntimeError as e:
    if USE_OPENAI_FALLBACK and client_openai and "OPENAI_FALLBACK" in str(e):
        print("  Tentando OpenAI fallback para SEO...")
        try:
            resp = client_openai.chat.completions.create(
                model=OPENAI_FALLBACK_MODEL,
                messages=[{"role": "user", "content": seo_prompt}],
                temperature=0.9
            )
            seo_content = resp.choices[0].message.content
            seo_model_used = f"OpenAI ({OPENAI_FALLBACK_MODEL})"
            print("  OpenAI fallback OK.")
        except Exception as oe:
            print(f"  OpenAI tambem falhou: {oe}")

except Exception as e:
    print(f"Erro no SEO: {e}")

if seo_content:
    final_report = f"""
============================================================
GUIA DE POSTAGEM — {current_anime_title}
Modelo: {seo_model_used}
============================================================

{seo_content}

============================================================
{short_data_report}
============================================================
"""
    print(final_report)

    with open(GUIA_PATH, "w", encoding="utf-8") as f:
        f.write(final_report)

    if 'salvar_no_drive' in globals():
        salvar_no_drive(GUIA_PATH, f"KAGGLE/AUDIO_DUB/{os.path.basename(GUIA_PATH)}")
else:
    print("Guia SEO nao gerado.")

cell_end(5, "done", "Identificacao do Anime concluido")

## Fase 4 - Clonagem Vocal


In [ ]:
cell_start(6, "Celula 6")
# @title 🔍 5.1 Verificação/Busca do Áudio de Clonagem
# @markdown Se a Célula 1 já encontrou e baixou o áudio de referência, apenas confirma.
# @markdown Caso contrário, busca no Drive usando NOME_NARRADOR.
import os
from difflib import SequenceMatcher

BASE_PATH  = globals().get('BASE_PATH', '/kaggle/working/AUDIO_DUB')

# Lê configurações da Cel1
nome_personagem = globals().get('NOME_NARRADOR', 'Alessandro')
usar_clonagem   = True
print(f"Narrador: {nome_personagem}")

# Se a Cel1 já baixou o áudio, usa direto
if globals().get('REF_AUDIO_PATH', '') and os.path.exists(globals().get('REF_AUDIO_PATH', '')):
    REF_AUDIO_PATH = globals()['REF_AUDIO_PATH']
    print(f"Referencia ja definida pela Cel1: {REF_AUDIO_PATH}")
else:
    REF_AUDIO_PATH = ""
    CLONAGEM_DIR_DRIVE = "KAGGLE/AUDIO_DUB/INPUT/CLONAGEM"
    LOCAL_CLONAGEM_DIR = f"{BASE_PATH}/INPUT/CLONAGEM"
    os.makedirs(LOCAL_CLONAGEM_DIR, exist_ok=True)

    def similaridade(a, b):
        return SequenceMatcher(None, a.lower(), b.lower()).ratio()

    if usar_clonagem and nome_personagem and 'drive_service' in globals() and drive_service:
        print(f"Buscando '{nome_personagem}' na pasta '{CLONAGEM_DIR_DRIVE}'...")
        parent_id = _buscar_id(CLONAGEM_DIR_DRIVE)
        if parent_id:
            query   = f"'{parent_id}' in parents and trashed=false and mimeType contains 'audio/'"
            results = drive_service.files().list(q=query, fields="files(id, name)").execute()
            arquivos = results.get('files', [])

            melhor_match = None
            maior_score  = 0.0
            for arq in arquivos:
                nome_limpo = os.path.splitext(arq['name'])[0]
                score = similaridade(nome_personagem, nome_limpo)
                if nome_personagem.lower() in nome_limpo.lower():
                    score = max(score, 0.85)
                if score > maior_score:
                    maior_score  = score
                    melhor_match = arq

            if melhor_match and maior_score > 0.6:
                print(f"  Match: '{melhor_match['name']}' ({maior_score*100:.1f}%)")
                caminho_local = f"{LOCAL_CLONAGEM_DIR}/{melhor_match['name']}"
                if not os.path.exists(caminho_local):
                    from googleapiclient.http import MediaIoBaseDownload
                    request = drive_service.files().get_media(fileId=melhor_match['id'])
                    with open(caminho_local, "wb") as f:
                        downloader = MediaIoBaseDownload(f, request)
                        done = False
                        while not done: _, done = downloader.next_chunk()
                    print("  Audio baixado!")
                REF_AUDIO_PATH = caminho_local
            else:
                print(f"  Nenhum audio com >60% de similaridade com '{nome_personagem}'.")
        else:
            print("  Pasta CLONAGEM nao encontrada no Drive.")

if not REF_AUDIO_PATH:
    print("Sem audio de clonagem. Usando TTS sintetica.")
else:
    print(f"Referencia definida: {REF_AUDIO_PATH}")

cell_end(6, "done", "Celula 6 concluido")

In [ ]:
cell_start(7, "Traducao Inteligente")
# @title 🎤 5.2 Transcrição do Áudio de Referência (Clonagem)
# @markdown Transcreve o áudio de referência para usar como prompt no OmniVoice.
import os

REF_TEXT = ""

if 'REF_AUDIO_PATH' in globals() and REF_AUDIO_PATH and os.path.exists(REF_AUDIO_PATH):
    txt_path = f"{REF_AUDIO_PATH}.txt"

    if os.path.exists(txt_path):
        print(f"✅ Transcrição em cache: {txt_path}")
        with open(txt_path, 'r', encoding='utf-8') as f:
            REF_TEXT = f.read().strip()
        print(f"📝 Texto: '{REF_TEXT}'")
    else:
        print(f"🚀 Transcrevendo: {os.path.basename(REF_AUDIO_PATH)}...")

        try:
            if 'stable_model' not in dir(): raise RuntimeError('Whisper nao carregou. Verifique MODEL_WHISPER_PATH.')
            result = stable_model.transcribe_stable(
                REF_AUDIO_PATH,
                beam_size=5,
                vad_filter=True
            )

            texto_completo = []
            for seg in result.segments:
                texto_completo.append(seg.text)

            REF_TEXT = " ".join(texto_completo).strip()

            if REF_TEXT:
                with open(txt_path, 'w', encoding='utf-8') as f:
                    f.write(REF_TEXT)
                print(f"✅ Transcrito e salvo!")
                print(f"📝 Texto: '{REF_TEXT}'")
            else:
                print("⚠️ Nenhuma fala detectada neste áudio.")
        except Exception as e:
            print(f"❌ Erro na transcrição: {e}")
else:
    print("ℹ️ Sem áudio de clonagem. Nenhuma transcrição necessária.")

cell_end(7, "done", "Traducao Inteligente concluido")

## Fase 5 - Servidores OmniVoice (GPU0 + GPU1)

Execute ANTES da Fabrica.


In [ ]:
cell_start(8, "Celula 8")
# @title 🎙️ 1b. Servidores OmniVoice (GPU0 + GPU1)
# @markdown Sobe um servidor por GPU disponível. Rode antes da Cel6 (Fábrica de Áudio).
import subprocess, urllib.request, time, os, threading

GPU_COUNT = globals().get('GPU_COUNT', torch.cuda.device_count() if 'torch' in dir() else 1)

print(f"GPUs disponíveis: {GPU_COUNT}")
print("Subindo servidores OmniVoice...")

# Corrige path do modelo OmniVoice no CLI
demo_file = "/usr/local/lib/python3.12/dist-packages/omnivoice/cli/demo.py"
if os.path.exists(demo_file):
    with open(demo_file, "r") as f:
        content = f.read()
    if '"k2-fsa/OmniVoice"' in content:
        content = content.replace('"k2-fsa/OmniVoice"', f'"{MODEL_OMNIVOICE_PATH}"')
        with open(demo_file, "w") as f:
            f.write(content)
        print("CLI OmniVoice corrigido para dataset local.")

_server_procs = []
OMNIVOICE_PORTS = []

for gpu_idx in range(min(GPU_COUNT, 2)):   # máximo 2 servidores
    port  = 8001 + gpu_idx
    log   = open(f"/kaggle/working/omnivoice_gpu{gpu_idx}.log", "w")
    env   = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu_idx)}
    proc  = subprocess.Popen(
        ["omnivoice-demo", "--ip", "127.0.0.1", "--port", str(port)],
        env=env, stdout=log, stderr=subprocess.STDOUT
    )
    _server_procs.append(proc)
    OMNIVOICE_PORTS.append(port)
    print(f"  GPU{gpu_idx} → porta {port} iniciando...")

def _aguardar_servidor(port, label):
    for i in range(40):
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/", timeout=2)
            print(f"  {label} online!")
            return True
        except:
            time.sleep(5)
            if (i + 1) % 4 == 0:
                print(f"  [{(i+1)*5}s] {label} subindo...")
    print(f"  {label} nao respondeu. Verifique o log.")
    return False

# Espera todos os servidores em paralelo
threads = [
    threading.Thread(target=_aguardar_servidor, args=(port, f"GPU{idx}:porta{port}"))
    for idx, port in enumerate(OMNIVOICE_PORTS)
]
for t in threads: t.start()
for t in threads: t.join()

print(f"\nServidores ativos: {OMNIVOICE_PORTS}")
print("Pronto para dublar!")

cell_end(8, "done", "Celula 8 concluido")

## Fase 6 - Fabrica de Audio

TTS paralelo GPU0+GPU1.


In [ ]:
cell_start(9, "Gerador Short + Guia")
# @title 🎙️ 6. Fábrica de Áudio: Dublagem (NARRACAO) + Recorte Original (CENA)
# @markdown Gera Video Completo E Short. A ordem e definida por TARGET_MODE_FIRST na Celula 1.
import os
import re
import json
import soundfile as sf
from tqdm.notebook import tqdm
from pydub import AudioSegment
from pydub.silence import detect_leading_silence

# --- Lê configurações globais da Célula 1 ---
TARGET_MODE_FIRST = globals().get('TARGET_MODE_FIRST', 'Video Completo')
TARGET_MIN_SPEED  = globals().get('TARGET_MIN_SPEED', 1.0)
TARGET_MAX_SPEED  = globals().get('TARGET_MAX_SPEED', 1.25)
MAX_RETRIES       = globals().get('MAX_AUDIO_RETRIES', 4)
MAX_TEXT_RETRIES  = globals().get('MAX_TEXT_RETRIES', 3)
BASE_PATH         = globals().get('BASE_PATH', '/kaggle/working/AUDIO_DUB')
TEMP_DIR          = globals().get('TEMP_PATH', f'{BASE_PATH}/TEMP')
ANIME_AUDIO_ORIGINAL = globals().get('ANIME_AUDIO_ORIGINAL', '')
current_protagonist  = globals().get('protagonist', 'Narrador')

os.makedirs(TEMP_DIR, exist_ok=True)

# Ordem de execução: o modo escolhido primeiro, depois o outro
_second_mode = "Short Viral" if TARGET_MODE_FIRST == "Video Completo" else "Video Completo"
MODOS_EXECUCAO = [TARGET_MODE_FIRST, _second_mode]

print(f"Ordem de geracao: 1) {TARGET_MODE_FIRST}  |  2) {_second_mode}")
print(f"Narrador: {current_protagonist} | 3a Pessoa")

# --- Voz de Clonagem ---
_ref_audio = globals().get('REF_AUDIO_PATH', '')
_ref_text  = globals().get('REF_TEXT', '')

if _ref_audio and _ref_audio.endswith('.m4a') and os.path.exists(_ref_audio):
    _ref_wav = _ref_audio.replace('.m4a', '.wav')
    if not os.path.exists(_ref_wav):
        AudioSegment.from_file(_ref_audio, format='m4a').export(_ref_wav, format='wav')
    _ref_audio = _ref_wav

if not _ref_audio or not os.path.exists(_ref_audio):
    print('Clonagem nao disponivel. Usando sintese padrao.')
    REF_AUDIO_PATH = ''
    REF_TEXT       = ''
else:
    REF_AUDIO_PATH = _ref_audio
    REF_TEXT       = _ref_text
    print(f'Clonagem ativa: {REF_AUDIO_PATH}')

# --- Áudio original do anime (para recortes CENA) ---
# Carregado aqui uma vez; reutilizado por recortar_cena() em ambos os modos.
anime_audio_full = None
if ANIME_AUDIO_ORIGINAL and os.path.exists(ANIME_AUDIO_ORIGINAL):
    anime_audio_full = AudioSegment.from_file(ANIME_AUDIO_ORIGINAL)
    print(f"Audio do anime carregado ({len(anime_audio_full)/1000:.1f}s)")
else:
    print("AVISO: Audio original nao encontrado. Blocos CENA ficarao silenciosos.")

# --- Clientes OmniVoice (um por GPU) ---
from gradio_client import Client, handle_file
import shutil, threading
from concurrent.futures import ThreadPoolExecutor

_portas = globals().get('OMNIVOICE_PORTS', []) or [8001]
omni_clients = []
for _p in _portas:
    try:
        omni_clients.append(Client(f"http://127.0.0.1:{_p}/"))
        print(f"OmniVoice conectado na porta {_p}")
    except Exception as _e:
        print(f"Porta {_p} offline: {_e}")

if not omni_clients:
    # fallback: tenta 8001
    try:
        omni_clients.append(Client("http://127.0.0.1:8001/"))
    except:
        print("AVISO: Nenhum servidor OmniVoice disponivel!")

_num_gpus = len(omni_clients)
print(f"  {_num_gpus} servidor(es) ativo(s) — TTS {'paralelo' if _num_gpus > 1 else 'sequencial'}")

# Lock para salvar progresso com segurança entre threads
_save_lock = threading.Lock()

# --- Funções de Áudio ---
def limpar_texto(texto):
    return re.sub(r'[*"(){}\[\]_~]', '', texto).strip()

def create_silent_audio(filename, duration_ms=1000):
    AudioSegment.silent(duration=duration_ms).export(filename, format='mp3')
    return duration_ms / 1000.0

def trim_audio_silence(wav_in, mp3_out, padding_ms=50):
    try:
        sound = AudioSegment.from_file(wav_in)
        st = detect_leading_silence(sound, -60.0, 10)
        en = detect_leading_silence(sound.reverse(), -60.0, 10)
        trimmed = sound[st:max(st + 100, len(sound) - en)]
        final = AudioSegment.silent(duration=padding_ms) + trimmed + AudioSegment.silent(duration=padding_ms)
        final.export(mp3_out, format='mp3')
        if os.path.exists(wav_in): os.remove(wav_in)
        return len(final) / 1000.0
    except:
        return 0.0

def generate_tts(text, filename, gpu_idx=0):
    texto = limpar_texto(text)
    if len(texto) < 2: return create_silent_audio(filename, 1000)
    # du = hint de duração para o OmniVoice. O loop de otimização corrige depois.
    est_duration = max(2.0, min(len(texto) / 13.0, 30.0))
    client = omni_clients[gpu_idx % len(omni_clients)] if omni_clients else None
    if not client:
        return create_silent_audio(filename, 1500)
    try:
        if REF_AUDIO_PATH and os.path.exists(REF_AUDIO_PATH):
            result = client.predict(
                text=texto, lang="Portuguese",
                ref_aud=handle_file(REF_AUDIO_PATH),
                ref_text=REF_TEXT if REF_TEXT else texto[:50],
                instruct="male, young adult",
                ns=32.0, gs=2.0, dn=True, sp=1.0, du=est_duration,
                pp=True, po=True, api_name="/_clone_fn"
            )
        else:
            result = client.predict(
                text=texto, lang="Portuguese",
                ns=32.0, gs=3.0, dn=True, sp=1.2, du=est_duration,
                pp=True, po=True,
                param_9="Male / 男", param_10="Young Adult / 青年",
                param_11="Moderate Pitch / 中音调",
                param_12="Auto", param_13="Auto", param_14="Auto",
                api_name="/_design_fn"
            )
        audio_path = result[0]
        if audio_path is None or not os.path.exists(audio_path):
            return create_silent_audio(filename, 1500)
        wav_local = filename.replace('.mp3', '.wav')
        shutil.copy(audio_path, wav_local)
        return trim_audio_silence(wav_local, filename)
    except Exception as e:
        print(f'Erro TTS (GPU{gpu_idx}): {e}')
        return create_silent_audio(filename, 1500)

def recortar_cena(block, filename):
    try:
        trecho = anime_audio_full[int(block['start'] * 1000):int(block['end'] * 1000)]
        trecho.export(filename, format='mp3')
        return len(trecho) / 1000.0
    except:
        return create_silent_audio(filename, int((block['end'] - block['start']) * 1000))

def get_context_window(block_list, current_id, before=4, after=4):
    """
    Retorna uma string com o contexto narrativo:
    - 'before' blocos anteriores (do tipo NARRACAO) mais próximos
    - bloco atual
    - 'after' blocos seguintes (do tipo NARRACAO) mais próximos
    """
    idx = next((i for i, b in enumerate(block_list) if b['id'] == current_id), 0)

    # Coleta anteriores
    prev_texts = []
    i = idx - 1
    while i >= 0 and len(prev_texts) < before:
        if block_list[i].get('tipo') == 'NARRACAO' and block_list[i].get('translated_text'):
            prev_texts.insert(0, block_list[i]['translated_text'])
        i -= 1

    # Coleta posteriores
    next_texts = []
    i = idx + 1
    while i < len(block_list) and len(next_texts) < after:
        if block_list[i].get('tipo') == 'NARRACAO' and block_list[i].get('translated_text'):
            next_texts.append(block_list[i]['translated_text'])
        i += 1

    parts = []
    if prev_texts:
        parts.append("CONTEXTO ANTERIOR: " + " ".join(prev_texts))
    parts.append("TEXTO ATUAL: " + (block_list[idx].get('translated_text') or ''))
    if next_texts:
        parts.append("CONTEXTO POSTERIOR: " + " ".join(next_texts))
    return "\n".join(parts)

# ──────────────────────────────────────────────────────────────────────────────
# FUNÇÃO PRINCIPAL DA FÁBRICA (reutilizada para cada modo)
# ──────────────────────────────────────────────────────────────────────────────
def run_fabrica(modo):
    is_short   = (modo == "Short Viral")
    prefix     = "short_" if is_short else "block_"
    save_name  = (
        f"{BASE_PATH}/ROTEIRO_ADAPTADO_SHORT_FINAL.json"
        if is_short else
        f"{BASE_PATH}/ROTEIRO_ADAPTADO_COMPLETO_FINAL.json"
    )

    print(f"\n{'='*60}")
    print(f"  INICIANDO: {modo}")
    print(f"{'='*60}")

    # --- Carregamento com Resume ---
    if os.path.exists(save_name):
        print(f"  Retomando cache: {save_name}")
        with open(save_name, 'r', encoding='utf-8') as f:
            target_list = json.load(f)
    else:
        src = (
            f"{BASE_PATH}/roteiro_short.json"
            if is_short else
            f"{BASE_PATH}/traducao_simplificada.json"
        )
        if not os.path.exists(src):
            print(f"  AVISO: Arquivo fonte nao encontrado ({src}). Pulando {modo}.")
            return
        with open(src, 'r', encoding='utf-8') as f:
            target_list = json.load(f)

    def salvar_progresso():
        with _save_lock:
            with open(save_name, 'w', encoding='utf-8') as f:
                json.dump(target_list, f, ensure_ascii=False, indent=4)
            if 'salvar_no_drive' in globals():
                salvar_no_drive(save_name, f'KAGGLE/AUDIO_DUB/{os.path.basename(save_name)}')

    # --- 1. Geração Inicial (V0) — paralela entre GPU0 e GPU1 ---
    pending = [
        (i, b) for i, b in enumerate(target_list)
        if not (b.get('status') == 'GREEN' and os.path.exists(b.get('current_file', '')))
    ]
    print(f"\n  Gerando audios V0: {len(pending)} blocos pendentes | {_num_gpus} GPU(s)")

    def _process_v0(args):
        enum_idx, block = args
        gpu_idx  = enum_idx % _num_gpus
        filename = os.path.join(TEMP_DIR, f"{prefix}{block['id']}_v0.mp3")
        orig_dur = max(0.1, block['end'] - block['start'])
        if block.get('tipo') == 'CENA':
            dur = recortar_cena(block, filename)
            block['current_file']     = filename
            block['current_duration'] = dur
            block['speed_factor']     = 1.0
            block['status']           = 'GREEN'
        else:
            dur   = generate_tts(block.get('translated_text', ''), filename, gpu_idx)
            ratio = dur / orig_dur if dur > 0 else 1.0
            block['current_file']     = filename
            block['current_duration'] = dur
            block['speed_factor']     = ratio
            block['status']           = 'GREEN' if TARGET_MIN_SPEED <= ratio <= TARGET_MAX_SPEED else 'RED'
        salvar_progresso()

    with ThreadPoolExecutor(max_workers=max(1, _num_gpus)) as ex:
        list(tqdm(ex.map(_process_v0, pending), total=len(pending), desc=f'V0 {modo[:5]}'))

    # --- 2. Loop de Otimização (PARALELIZADO) ---
    def _process_correction(args):
        """Processa um bloco RED: tenta reescrita e gera áudio, usando a GPU especificada."""
        enum_idx, block = args
        gpu_idx = enum_idx % _num_gpus

        original_dur = max(0.1, block['end'] - block['start'])
        if original_dur < 1.5:
            block['status'] = 'WARNING_SHORT'
            salvar_progresso()
            return

        current_len   = len(block['translated_text'])
        current_speed = max(0.1, block['speed_factor'])
        target_avg    = (TARGET_MIN_SPEED + TARGET_MAX_SPEED) / 2
        ideal_len     = int(current_len * (target_avg / current_speed))
        REAL_MIN      = max(5, int(ideal_len * 0.90))
        REAL_MAX      = int(ideal_len * 1.10)

        # NOVO: contexto expandido (4 anteriores, 4 seguintes)
        contexto = get_context_window(target_list, block['id'], before=4, after=4)
        prompt_min = REAL_MIN
        prompt_max = REAL_MAX
        valid_text_found = False
        candidate  = ''

        print(f"\n  Bloco {block['id']} | Speed: {current_speed:.2f}x | Meta: {REAL_MIN}-{REAL_MAX} chars")

        for attempt in range(1, MAX_TEXT_RETRIES + 1):
            prompt = (
                f"{contexto}\n"
                f"MISSAO: Reescreva o TEXTO ATUAL EXATAMENTE entre {prompt_min} e {prompt_max} caracteres, "
                f"mantendo a coerência com o CONTEXTO ANTERIOR e POSTERIOR. "
                f"Mantenha em 3a PESSOA (Ele/Ela). Nunca use 1a pessoa.\n"
                f"Responda APENAS o novo texto."
            )

            try:
                resp = gemini_generate(
                    model_chain=MODELS_REESCRITA,
                    contents=prompt,
                    task_name=f"Reescrita-{block['id']}-T{attempt}"
                )
                candidate = resp.text.strip().replace('"', '')

            except RuntimeError as e:
                if USE_OPENAI_FALLBACK and client_openai and "OPENAI_FALLBACK" in str(e):
                    try:
                        r = client_openai.chat.completions.create(
                            model=OPENAI_FALLBACK_MODEL,
                            messages=[
                                {'role': 'system', 'content': 'Contador restrito de letras.'},
                                {'role': 'user',   'content': prompt}
                            ],
                            temperature=0.7
                        )
                        candidate = r.choices[0].message.content.strip().replace('"', '')
                    except Exception as oe:
                        print(f'  OpenAI fallback falhou: {oe}')
                        break
                else:
                    print(f'  Erro fatal: {e}')
                    break

            except Exception as e:
                print(f'  Erro API: {e}')
                break

            lc = len(candidate)
            if REAL_MIN <= lc <= REAL_MAX:
                print(f"  [OK] T{attempt}: {lc} chars")
                valid_text_found = True
                break
            else:
                print(f"  [FORA] T{attempt}: {lc} chars. Calibrando...")
                if lc > REAL_MAX:
                    exc = lc - REAL_MAX
                    prompt_max = max(10, prompt_max - exc - 5)
                    prompt_min = max(5, prompt_min - exc - 5)
                else:
                    dif = REAL_MIN - lc
                    prompt_min += dif + 5
                    prompt_max += dif + 5

        if not valid_text_found:
            print(f"  Ignorando Bloco {block['id']} — falhou em {MAX_TEXT_RETRIES} tentativas.")
            salvar_progresso()
            return

        block['translated_text'] = candidate
        new_file = os.path.join(TEMP_DIR, f"{prefix}{block['id']}_v{iteration}.mp3")
        dur   = generate_tts(candidate, new_file, gpu_idx=gpu_idx)
        ratio = dur / original_dur if dur > 0 else 1.0

        block['current_file']     = new_file
        block['current_duration'] = dur
        block['speed_factor']     = ratio
        block['status']           = 'GREEN' if TARGET_MIN_SPEED <= ratio <= TARGET_MAX_SPEED else 'RED'

        print(f"  Speed: {ratio:.2f}x -> {block['status']}")
        salvar_progresso()

    red_list  = [b for b in target_list if b.get('status') == 'RED' and b.get('tipo') == 'NARRACAO']
    iteration = 0

    while red_list and iteration < MAX_RETRIES:
        iteration += 1
        print(f"\n  Loop Audio [{modo[:5]}] - Iteracao {iteration}/{MAX_RETRIES} | {len(red_list)} blocos fora do limite.")

        correction_tasks = list(enumerate(red_list))

        with ThreadPoolExecutor(max_workers=max(1, _num_gpus)) as ex:
            list(tqdm(ex.map(_process_correction, correction_tasks),
                      total=len(correction_tasks),
                      desc=f'Correção {modo[:5]}'))

        red_list = [b for b in target_list if b.get('status') == 'RED' and b.get('tipo') == 'NARRACAO']

    print(f"\n  FIM [{modo}]: {len(red_list)} blocos irredutíveis restantes.")
    salvar_progresso()

# ──────────────────────────────────────────────────────────────────────────────
# EXECUÇÃO: roda ambos os modos na ordem definida por TARGET_MODE_FIRST
# ──────────────────────────────────────────────────────────────────────────────
for _modo in MODOS_EXECUCAO:
    run_fabrica(_modo)

print("\nFABRICA CONCLUIDA para ambos os modos!")

cell_end(9, "done", "Gerador Short + Guia concluido")

## Fase 7 - Montagem Final


In [ ]:
cell_start(10, "Celula 10")
# @title 🎹 7. Montagem Final + SRT Híbrido
# @markdown Monta Video Completo E Short na ordem definida por TARGET_MODE_FIRST.
import os, gc, json, glob, ffmpeg
from pydub import AudioSegment
from tqdm.notebook import tqdm

BASE_PATH  = globals().get('BASE_PATH', '/kaggle/working/AUDIO_DUB')
TEMP_DIR   = globals().get('TEMP_PATH', f'{BASE_PATH}/TEMP')
TARGET_MODE_FIRST = globals().get('TARGET_MODE_FIRST', 'Video Completo')
_second_mode = "Short Viral" if TARGET_MODE_FIRST == "Video Completo" else "Video Completo"
MODOS_MONTAGEM = [TARGET_MODE_FIRST, _second_mode]

current_anime_title = globals().get('selected_anime', 'Anime_Desconhecido')
safe_anime = current_anime_title.replace(' ', '_').replace('/', '_')

def fmt_ts(s):
    ms = int((s - int(s)) * 1000)
    m, sec = divmod(int(s), 60)
    h, m = divmod(m, 60)
    return f"{h:02d}:{m:02d}:{sec:02d},{ms:03d}"

def run_montagem(modo):
    is_short   = (modo == "Short Viral")
    prefix     = "short_" if is_short else "block_"
    json_final = (
        f"{BASE_PATH}/ROTEIRO_ADAPTADO_SHORT_FINAL.json"
        if is_short else
        f"{BASE_PATH}/ROTEIRO_ADAPTADO_COMPLETO_FINAL.json"
    )
    modo_folder = 'Short' if is_short else 'Completo'

    print(f"\n{'='*60}")
    print(f"  MONTAGEM: {modo}")
    print(f"{'='*60}")

    # --- Carregamento ---
    if os.path.exists(json_final):
        with open(json_final, 'r', encoding='utf-8') as f:
            lista_render = json.load(f)
        print(f"  {len(lista_render)} blocos carregados de {os.path.basename(json_final)}")
    elif modo == "Short Viral" and os.path.exists(f"{BASE_PATH}/roteiro_short.json"):
        with open(f"{BASE_PATH}/roteiro_short.json", 'r', encoding='utf-8') as f:
            lista_render = json.load(f)
        print(f"  Usando roteiro_short.json ({len(lista_render)} blocos)")
    elif not is_short and os.path.exists(f"{BASE_PATH}/traducao_simplificada.json"):
        with open(f"{BASE_PATH}/traducao_simplificada.json", 'r', encoding='utf-8') as f:
            lista_render = json.load(f)
        print(f"  Usando traducao_simplificada.json ({len(lista_render)} blocos)")
    else:
        print(f"  AVISO: Nenhum JSON encontrado para '{modo}'. Pulando.")
        return

    FINAL_OUTPUT_DIR = f"{BASE_PATH}/{safe_anime}/{modo_folder}"
    os.makedirs(FINAL_OUTPUT_DIR, exist_ok=True)

    OUTPUT_FILENAME = (
        f"{FINAL_OUTPUT_DIR}/SHORT_VIRAL_FINAL.mp3"
        if is_short else
        f"{FINAL_OUTPUT_DIR}/VIDEO_COMPLETO_FINAL.mp3"
    )
    SRT_FILENAME = OUTPUT_FILENAME.replace('.mp3', '.srt')

    # --- Canvas ---
    start_offset = lista_render[0]['start']
    total_ms = int((lista_render[-1]['end'] - start_offset) * 1000) + 2000
    final_mix = AudioSegment.silent(duration=total_ms)

    # --- Colagem ---
    print(f"  Montando {len(lista_render)} blocos...")
    for block in tqdm(lista_render, desc=f'Mixing {modo[:5]}'):
        block_id = block['id']
        padrao   = os.path.join(TEMP_DIR, f"{prefix}{block_id}_v*.mp3")
        arquivos = glob.glob(padrao)
        if not arquivos:
            continue
        audio_file = max(arquivos, key=lambda x: int(x.split('_v')[-1].split('.mp3')[0]))
        seg   = AudioSegment.from_file(audio_file)
        ratio = block.get('speed_factor', 1.0)
        tipo  = block.get('tipo', 'NARRACAO')

        if tipo == 'NARRACAO':
            safe_speed = max(0.6, min(ratio, 1.6))
            if safe_speed != 1.0:
                tmp_wav  = f"tmp_mix_{block_id}.wav"
                proc_wav = f"proc_mix_{block_id}.wav"
                seg.export(tmp_wav, format='wav')
                try:
                    (
                        ffmpeg.input(tmp_wav)
                        .filter('atempo', safe_speed)
                        .output(proc_wav, acodec='pcm_s16le', ac=1, ar='44100', loglevel='quiet')
                        .overwrite_output().run()
                    )
                    if os.path.exists(proc_wav):
                        seg = AudioSegment.from_wav(proc_wav)
                        os.remove(proc_wav)
                except Exception as e:
                    print(f'  Erro atempo bloco {block_id}: {e}')
                finally:
                    if os.path.exists(tmp_wav): os.remove(tmp_wav)

        pos_ms = max(0, int((block['start'] - start_offset) * 1000))
        final_mix = final_mix.overlay(seg, position=pos_ms)

    print(f'  Exportando: {OUTPUT_FILENAME}')
    final_mix.export(OUTPUT_FILENAME, format='mp3')
    del final_mix
    gc.collect()
    print(f'  MP3 salvo.')

    # --- SRT Híbrido ---
    print(f'  Gerando SRT...')
    srt_entries = []
    for block in tqdm(lista_render, desc='SRT'):
        tipo     = block.get('tipo', 'NARRACAO')
        block_id = block['id']
        if tipo == 'NARRACAO':
            arquivos = glob.glob(os.path.join(TEMP_DIR, f"{prefix}{block_id}_v*.mp3"))
            if not arquivos: continue
            audio_file = max(arquivos, key=lambda x: int(x.split('_v')[-1].split('.mp3')[0]))
            try:
                if 'stable_model' in globals():
            if 'stable_model' not in dir(): raise RuntimeError('Whisper nao carregou. Verifique MODEL_WHISPER_PATH.')
                    result = stable_model.transcribe_stable(audio_file, language='pt', word_timestamps=True)
                    time_offset = block['start'] - start_offset
                    for seg in result.segments:
                        if hasattr(seg, 'words') and seg.words:
                            for word in seg.words:
                                srt_entries.append({
                                    'start': time_offset + word.start,
                                    'end':   time_offset + word.end,
                                    'text':  word.word.strip()
                                })
                else:
                    raise Exception("stable_model nao carregado.")
            except Exception:
                srt_entries.append({
                    'start': block['start'] - start_offset,
                    'end':   block['end']   - start_offset,
                    'text':  block.get('translated_text', '')
                })
        elif tipo == 'CENA' and block.get('translated_text', '').strip():
            srt_entries.append({
                'start': block['start'] - start_offset,
                'end':   block['end']   - start_offset,
                'text':  block['translated_text']
            })

    srt_entries.sort(key=lambda x: x['start'])
    with open(SRT_FILENAME, 'w', encoding='utf-8') as srt_file:
        for idx, entry in enumerate(srt_entries, 1):
            srt_file.write(f"{idx}\n")
            srt_file.write(f"{fmt_ts(entry['start'])} --> {fmt_ts(entry['end'])}\n")
            srt_file.write(f"{entry['text']}\n\n")
    print(f'  SRT: {len(srt_entries)} entradas salvas.')

    # --- Backup no Drive ---
    if 'salvar_no_drive' in globals():
        drive_mp3 = f"KAGGLE/AUDIO_DUB/OUTPUT/{safe_anime}_{modo_folder}.mp3"
        drive_srt = f"KAGGLE/AUDIO_DUB/OUTPUT/{safe_anime}_{modo_folder}.srt"
        try:
            salvar_no_drive(OUTPUT_FILENAME, drive_mp3)
            salvar_no_drive(SRT_FILENAME,    drive_srt)
            print(f"  Backup Drive: {drive_mp3}")
        except Exception as e:
            print(f"  Erro backup Drive: {e}")

# ─────────────────────────────────────────────────────────────────────────────
# EXECUÇÃO: ambos os modos na ordem TARGET_MODE_FIRST → outro
# ─────────────────────────────────────────────────────────────────────────────
print(f"Ordem de montagem: 1) {TARGET_MODE_FIRST}  |  2) {_second_mode}")
for _modo in MODOS_MONTAGEM:
    run_montagem(_modo)

print("\nMONTAGEM CONCLUIDA para ambos os modos!")

cell_end(10, "done", "Celula 10 concluido")

## Fase 8 - Limpeza


In [ ]:
cell_start(11, "Busca Audio Clonagem")
# @title 🧹 8. Faxina Cirúrgica (Painel de Controle)
# @markdown Selecione exatamente o que você deseja remover para liberar espaço ou refazer etapas.

import os
import shutil
import glob

# --- PAINEL DE CONTROLE ---
DELETE_TRANSCRIPTION = True # @param {type:"boolean"}
DELETE_TRANSLATION = True # @param {type:"boolean"}
DELETE_IDENTIFICATION = True # @param {type:"boolean"}
DELETE_SHORT_DATA = True # @param {type:"boolean"}
DELETE_AUDIO_CACHE = True # @param {type:"boolean"}
DELETE_FINAL_FILES = True # @param {type:"boolean"}

print("🧹 Iniciando Protocolo de Limpeza Seletiva...")

BASE_PATH = globals().get('BASE_PATH', '/kaggle/working/AUDIO_DUB')

# 1. Definição dos Alvos com BASE_PATH correto
targets = {
    "transcription": [f"{BASE_PATH}/transcricao_raw.json"],
    "translation":   [f"{BASE_PATH}/traducao_simplificada.json",
                      f"{BASE_PATH}/traducao_feita.json"],
    "identification":[f"{BASE_PATH}/identificacao_anime.json"],
    "short_data":    [f"{BASE_PATH}/GUIA_VIRAL_*.txt",
                      f"{BASE_PATH}/GUIA_POSTAGEM_*.txt",
                      f"{BASE_PATH}/roteiro_short.json",
                      f"{BASE_PATH}/ROTEIRO_ADAPTADO_SHORT_FINAL.json"],
    "final_files":   [f"{BASE_PATH}/**/SHORT_VIRAL_FINAL.mp3",
                      f"{BASE_PATH}/**/VIDEO_COMPLETO_FINAL.mp3",
                      f"{BASE_PATH}/**/DUB_FINAL_*.mp3",
                      f"{BASE_PATH}/ROTEIRO_ADAPTADO_COMPLETO_FINAL.json"],
}

# 2. Contador de Remoções
deleted_count = 0

# --- FUNÇÃO DE REMOÇÃO ---
def remove_pattern(pattern_list):
    count = 0
    for pat in pattern_list:
        # recursive=True para pegar arquivos em subpastas (**/)
        for f in glob.glob(pat, recursive=True):
            try:
                os.remove(f)
                count += 1
            except Exception as e:
                print(f"   Erro ao apagar {f}: {e}")
    return count

# --- EXECUÇÃO ---

# 1. Transcrição (Raw)
if DELETE_TRANSCRIPTION:
    n = remove_pattern(targets["transcription"])
    if n > 0: print(f"🗑️ Transcrição Original removida ({n} arquivos).")

# 2. Tradução (Base)
if DELETE_TRANSLATION:
    n = remove_pattern(targets["translation"])
    if n > 0: print(f"🗑️ Tradução Base removida ({n} arquivos).")

# 3. Identificação do Anime
if DELETE_IDENTIFICATION:
    n = remove_pattern(targets["identification"])
    if n > 0: print(f"🗑️ Identificação do anime removida ({n} arquivos).")

# 4. Dados do Short e Guias
if DELETE_SHORT_DATA:
    n = remove_pattern(targets["short_data"])
    # Resetar variáveis de memória também é importante
    if 'short_blocks' in globals():
        short_blocks = []
        print("   🧠 Memória 'short_blocks' limpa.")
    if n > 0: print(f"🗑️ Guias e Shorts removidos ({n} arquivos).")

# 5. Arquivos Finais (MP3 Prontos)
if DELETE_FINAL_FILES:
    n = remove_pattern(targets["final_files"])
    if n > 0: print(f"🗑️ Arquivos Finais (.mp3) removidos ({n} arquivos).")

# 6. Cache de Áudio (Pasta Temporária)
if DELETE_AUDIO_CACHE:
    temp_dir = globals().get('TEMP_PATH', '/kaggle/working/temp_audio')
    demucs_dir = "/kaggle/working/demucs_out"
    for d in [temp_dir, demucs_dir]:
        if os.path.exists(d):
            try:
                shutil.rmtree(d)
                os.makedirs(d, exist_ok=True)
                print(f"🗑️ {d} limpo e recriado.")
            except Exception as e:
                print(f"❌ Erro na pasta {d}: {e}")

# Resumo Final
print("-" * 40)
print("✨ Limpeza concluída conforme solicitado.")

cell_end(11, "done", "Busca Audio Clonagem concluido")